# Publication copy: portable paths and cleared outputs

Run the path configuration below before the original notebook cells. Change only PROJECT_ROOT. On Windows use a raw string such as r"D:\Phishing_Paper_Reproducibility". The data filenames and results/revision layout are preserved.

This copy has cleared cell outputs and execution counts. Keep the executed original and its saved results separately. This path revision is not a new scientific run or a full methodological audit.


In [ ]:
# Publication path configuration: edit only PROJECT_ROOT.
from pathlib import Path, PurePosixPath
import sys

# Empty: infer the root when opened from the project root or notebooks/.
# Otherwise enter your local absolute folder, or a /content/drive/... folder.
PROJECT_ROOT = ""
if not PROJECT_ROOT:
    candidates = [Path.cwd(), Path.cwd().parent]
    found = [p for p in candidates if (p / "data").is_dir() and (p / "notebooks").is_dir()]
    if len(found) != 1:
        raise RuntimeError("Set PROJECT_ROOT to your Phishing_Paper_Reproducibility folder.")
    PROJECT_ROOT = found[0]
PROJECT_ROOT = Path(PROJECT_ROOT).expanduser()
if str(PROJECT_ROOT).startswith("/content/drive/"):
    if "google.colab" not in sys.modules:
        raise RuntimeError("A Drive root requires Colab. Use a local folder on your computer.")
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive
        drive.mount("/content/drive")
PROJECT_ROOT = PROJECT_ROOT.resolve()
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f"Project root missing: {PROJECT_ROOT}")

DATA_FILENAMES = {
    "phiusiil.csv", "phiusiil_unified.csv", "URL-Phish.csv",
    "urlphish_unified.csv", "LegitPhish Dataset.csv", "legitphish_unified.csv",
}
def relocated_dataset_path(recorded_path):
    # Immutable historical records are kept byte-for-byte. Map only the
    # approved data members to this root; existing old Drive files are ignored.
    old = PurePosixPath(str(recorded_path).replace("\\", "/"))
    if old.parent.name != "data" or old.name not in DATA_FILENAMES:
        raise ValueError(f"Unrecognized recorded dataset: {recorded_path}")
    return PROJECT_ROOT / "data" / old.name

print("Project root:", PROJECT_ROOT)


# URL phishing pipeline — exact URL grouping revision

Reads the existing corrected full feature CSVs and the six approved preparation
files. No feature extraction or dataset CSV rewriting occurs. The source split
is fixed at 150,931 / 37,716 / 47,148 rows. Main target selections are URL-Phish
116,126 rows (99,526 legitimate / 16,600 phishing) and LegitPhish 71,767 rows
(8,089 legitimate / 63,678 phishing). These sizes are checked from raw URLs.

**Overlap-review correction / E1 / E2 / E4 / M1 / M2 / M6:** exact URL groups
define splits and uncertainty units. Target labels cannot cross adaptation and
evaluation through duplicate URLs. Label budgets count distinct URL labels.
MMD and entropy use a common restricted source-test reference absent from both
target URL sets; its changed population and class mix must be reported.

Run in Google Colab from top to bottom. The setup cell imports the installed
environment without upgrading preloaded NumPy or Matplotlib. Inputs, preparation
files and versions are fingerprinted. Old run directories remain unchanged.
All outputs go to a new results/revision run directory. Numerical completion
does not certify submission readiness or upstream feature definitions.


## 1. Environment and reproducibility

Managed notebook kernels can preload NumPy or Matplotlib before the first
user cell. Installing packages after that point leaves the old modules in
memory. The following cell checks that every required library imports; it
does not change the live kernel. The manifest records both **loaded runtime
versions** and installed distribution versions. The run writes reproducible
requirements from the loaded versions, plus separate reference pins. Use a
synthetic smoke run first in a new environment. **M6**: publication and a real
DOI are separate author actions.

In [ ]:
print("COLAB PIPELINE: exact-URL-grouped-v4; no package installation")
import importlib as _importlib
_required_modules={'numpy':'numpy','pandas':'pandas','scipy':'scipy',
                   'scikit-learn':'sklearn','lightgbm':'lightgbm',
                   'matplotlib':'matplotlib','joblib':'joblib',
                   'threadpoolctl':'threadpoolctl'}
_loaded={}
for _distribution,_module in _required_modules.items():
    try:
        _loaded[_distribution]=_importlib.import_module(_module).__version__
    except ImportError as _exc:
        raise RuntimeError(f'Missing/incompatible {_distribution}: {_exc}') from _exc
print('Loaded analysis libraries:',_loaded)

In [ ]:
"""Fresh-run URL benchmark pipeline. Labels: 0 legitimate, 1 phishing.

Every table is regenerated from the supplied unified feature CSVs; cached
notebook outputs are not inputs. See PROTOCOL.md in the run directory for estimands and limitations.
"""
from __future__ import annotations
import hashlib
import csv
import importlib.metadata as metadata
import json
import math
import os
import platform
import re
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import scipy
from scipy import sparse
from scipy.special import expit
from scipy.spatial.distance import cdist, pdist
from scipy.stats import beta, ks_2samp
import matplotlib.pyplot as plt
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, f1_score, roc_auc_score,
                             precision_recall_curve)
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.exceptions import ConvergenceWarning
import threadpoolctl

FEATURES = ['URLLength', 'DomainLength', 'IsDomainIP', 'TLD', 'TLDLength',
            'NoOfSubDomain', 'NoOfLettersInURL', 'LetterRatioInURL',
            'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL',
            'NoOfQMarkInURL', 'NoOfAmpersandInURL',
            'NoOfOtherSpecialCharsInURL', 'IsHTTPS']
NUMERIC = [c for c in FEATURES if c != 'TLD']
LENGTH = ['URLLength', 'DomainLength', 'TLDLength',
          'LetterRatioInURL', 'DegitRatioInURL']
COUNT = ['NoOfSubDomain', 'NoOfLettersInURL', 'NoOfDegitsInURL',
         'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL',
         'NoOfOtherSpecialCharsInURL']
DOMAIN = ['DomainLength', 'IsDomainIP', 'TLDLength', 'NoOfSubDomain', 'IsHTTPS']
SOURCE, TARGETS = 'PhiUSIIL', ['URL-Phish', 'LegitPhish']
PACKAGES = {'numpy': '2.3.5', 'pandas': '2.2.3', 'scipy': '1.17.0',
            'scikit-learn': '1.8.0', 'lightgbm': '4.6.0',
            'matplotlib': '3.10.8', 'joblib': '1.5.3',
            'threadpoolctl': '3.6.0'}

# FIX A08: Notebook cells share mutable state. Enforce the same order as the
# standalone runner, so a completed run cannot be silently overwritten by a
# repeated earlier cell while its manifest still claims completion.
STAGE_ORDER = ['stage_inputs','stage_splits_and_models',
    'stage_baseline_and_operating_points','stage_calibration','stage_label_budgets',
    'stage_shift_and_pruning','stage_lofo','stage_mmd','stage_shap',
    'stage_entropy_and_risk','stage_figures','stage_finish']

def validate_config(config):
    """FIX A06: Reject invalid/ambiguous settings before allocating a run folder."""
    required=set(default_config())
    missing=required-set(config)
    if missing:
        raise ValueError(f'Missing configuration keys: {sorted(missing)}')
    if config['mode'] not in ('full','smoke'):
        raise ValueError('mode must be full or smoke')
    if set(config['datasets'])!={SOURCE,*TARGETS}:
        raise ValueError('Configure exactly PhiUSIIL, URL-Phish and LegitPhish')
    integer_keys=['bootstrap','repeats','mmd_n','mmd_repeats','mmd_permutations',
        'mmd_bootstrap','mmd_inference_n','mmd_median_n','shap_source_n',
        'shap_error_n','window_size','monitor_windows']
    def integer(v,minimum):
        return isinstance(v,int) and not isinstance(v,bool) and v>=minimum
    for key in integer_keys:
        if not integer(config[key],2):
            raise ValueError(f'{key} must be an integer >= 2')
    for key in ['seed','threads']:
        if not integer(config[key],1 if key=='threads' else 0):
            raise ValueError(f'{key} must be a valid nonnegative integer (threads >= 1)')
    seeds=config['lofo_seeds']
    if not isinstance(seeds,list) or len(seeds)<2 or not all(integer(s,0) for s in seeds) or len(set(seeds))!=len(seeds):
        raise ValueError('Use at least two distinct nonnegative integer LOFO seeds')
    for key,low,high in [('budget_fractions',0.,.5),('fpr_budgets',0.,1.)]:
        values=config[key]
        if not isinstance(values,list) or not values:
            raise ValueError(f'{key} must be a nonempty list')
        if not all(isinstance(v,(int,float)) and not isinstance(v,bool) and math.isfinite(v)
                   and (low<v<=high if key=='budget_fractions' else low<=v<=high) for v in values):
            raise ValueError(f'Invalid {key}: label fractions in (0,0.5]; FPR budgets in [0,1]')
        if len(set(values))!=len(values):
            raise ValueError(f'Duplicate {key} would duplicate results')
    ks=config['pruning_k']
    if not isinstance(ks,list) or not ks or not all(integer(k,0) for k in ks) or len(set(ks))!=len(ks):
        raise ValueError('pruning_k must contain distinct nonnegative integers')
    for domain,spec in config['datasets'].items():
        if not isinstance(spec.get('path'),str) or not spec['path'].strip():
            raise ValueError(f'{domain}: unified CSV path must be nonempty')
        counts=spec.get('expected_counts')
        if counts is not None and (not isinstance(counts,dict) or set(counts)!={'0','1'} or
                                  not all(integer(v,1) for v in counts.values())):
            raise ValueError(f'{domain}: invalid expected harmonized counts')
    if not isinstance(config['strict_versions'],bool):
        raise ValueError('strict_versions must be boolean')
    for key in ['data_dir','output_root']:
        if not isinstance(config[key],str) or not config[key].strip():
            raise ValueError(f'{key} must be a nonempty path string')

def default_config():
    # E1/R1-1/R2-4: These are ALREADY unified files; label 0 means legitimate
    # and 1 phishing. Never invert labels or extract URLs in this notebook.
    return {
        'data_dir': './data', 'output_root': './results/revision',
        'preparation_dir': '',
        # Approved preparation digests; raw input digests are checked through these records.
        'preparation_sha256': {
            "legitphish_selection.npz": "28b278e71be2084c4a2ed301cadd57eeff5bd092212dd8badf363f7c797bd295",
            "source_partitions.npz": "ee737850744aabdef98b7c8e5754c6785b9d421c61bb40e5203237f94dbadf5f",
            "urlphish_selection.npz": "d5f755045175b9fa8b3ed6c0924024deb5daa1080c00e7fd62fc1d03a6d536ec",
            "legitphish_selection_record.json": "04ad3ca5f9811441f08685dfa6f2685e3a5f8480ffd74d43d0db96e427e4d636",
            "source_partition_record.json": "bc79e45b87b10e3dacd118b4981b07b8ef8fa3c70c6bc82016b507abac85a3ad",
            "urlphish_selection_record.json": "462a3701362786bc29028afcc5eb4757353e38170ef0206cc88b90ca45087173"
},
        'mode': 'full', 'seed': 42, 'threads': 4, 'strict_versions': True,
        'bootstrap': 1000, 'repeats': 20, 'lofo_seeds': [42,43,44,45,46],
        'mmd_n': 3000, 'mmd_repeats': 20, 'mmd_permutations': 200,
        'mmd_bootstrap': 1000, 'mmd_inference_n': 1000,
        'mmd_median_n': 2000, 'shap_source_n': 3000, 'shap_error_n': 10000,
        'window_size': 500, 'monitor_windows': 1000,
        # R1-2: target budgets are fractions of distinct URLs in each evaluated cohort. On the
        # smaller LegitPhish cohort, 0.1% is 71 distinct URL labels, not 100.
        # Include the original percent-scale budgets as well as
        # the original percent-scale budgets. Exact realized counts are saved.
        'budget_fractions': [0.001,0.003,0.01,0.05,0.10,0.20],
        'fpr_budgets': [0.001,0.01,0.05],
        'pruning_k': [0,10,50,100,250,500],
        'datasets': {
            SOURCE: {'path':'phiusiil_unified.csv',
                     'expected_counts':{'0':134850,'1':100945}},
            'URL-Phish': {'path':'urlphish_unified.csv',
                          'expected_counts':{'0':100000,'1':16600},
                          'selected_counts':{'0':99526,'1':16600}},
            'LegitPhish': {'path':'legitphish_unified.csv',
                           'expected_counts':{'0':37540,'1':63678},
                           'selected_counts':{'0':8089,'1':63678},
                           'raw':'LegitPhish Dataset.csv'}
        }
    }

def smoke_config(data_dir, output_root):
    """Reduced synthetic-test workload; never a publication run."""
    c = default_config()
    c.update(data_dir=str(data_dir), output_root=str(output_root), mode='smoke',
             strict_versions=False, bootstrap=24, repeats=2, lofo_seeds=[42, 43],
             mmd_n=40, mmd_repeats=2, mmd_permutations=19, mmd_bootstrap=24,
             mmd_inference_n=40, mmd_median_n=80, shap_source_n=80,
             shap_error_n=80, window_size=25, monitor_windows=24,
             budget_fractions=[0.05, 0.10, 0.20], pruning_k=[0, 10, 50])
    for d in c['datasets'].values():
        d['expected_counts'] = None
    # The synthetic test has no corrected raw files or URL audit record.
    for field in ('row_record','corrected_full','corrected_raw','source_raw'):
        c['datasets']['LegitPhish'].pop(field, None)
    c['datasets']['LegitPhish']['path']='legitphish_unified.csv'
    return c

def make_smoke_inputs(folder):
    """Synthetic unified features for execution checks; never publication results."""
    folder=Path(folder).expanduser().resolve()
    folder.mkdir(parents=True,exist_ok=True)
    rng=np.random.default_rng(390)
    for domain,n,rate,filename in [(SOURCE,1800,.44,'phiusiil_unified.csv'),
            ('URL-Phish',700,.30,'urlphish_unified.csv'),
            ('LegitPhish',700,.63,'legitphish_unified.csv')]:
        y=np.array([0]*(n-int(n*rate))+[1]*int(n*rate))
        rng.shuffle(y)
        length=rng.integers(25,110,n)
        letters=np.minimum(length,rng.integers(12,60,n))
        digits=np.minimum(length-letters,rng.integers(0,16,n))
        tld=rng.choice(['com','org','edu','net','missing','xyz'],n)
        df=pd.DataFrame({
            'URLLength':length, 'DomainLength':rng.integers(5,25,n),
            'IsDomainIP':rng.binomial(1,.03,n), 'TLD':tld,
            'TLDLength':[0 if x=='missing' else len(x) for x in tld],
            'NoOfSubDomain':rng.integers(0,4,n),
            'NoOfLettersInURL':letters,'LetterRatioInURL':letters/length,
            'NoOfDegitsInURL':digits,'DegitRatioInURL':digits/length,
            'NoOfEqualsInURL':rng.integers(0,3,n),
            'NoOfQMarkInURL':rng.integers(0,2,n),
            'NoOfAmpersandInURL':rng.integers(0,3,n),
            'NoOfOtherSpecialCharsInURL':rng.integers(0,10,n),
            'IsHTTPS':rng.binomial(1,np.where(y==1,.70,.45)),
            'label':y})
        df.to_csv(folder/filename,index=False)
    return folder

def digest_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1024 * 1024), b''):
            h.update(b)
    return h.hexdigest()

def jsonable(x):
    if isinstance(x, dict):
        return {str(k): jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [jsonable(v) for v in x]
    if isinstance(x, np.ndarray):
        return jsonable(x.tolist())
    if isinstance(x, np.generic):
        return jsonable(x.item())
    if isinstance(x, Path):
        return str(x)
    if isinstance(x, float) and not math.isfinite(x):
        return 'infinity' if x > 0 else '-infinity' if x < 0 else None
    return x

class RunState:
    def __init__(self, config):
        validate_config(config)
        self.cfg = json.loads(json.dumps(config))
        if self.cfg['mode'] not in ('full', 'smoke'):
            raise ValueError('mode must be full or smoke')
        if set(self.cfg['datasets'])!={SOURCE,*TARGETS}:
            raise ValueError('Configure exactly PhiUSIIL, URL-Phish and LegitPhish')
        for key in ['bootstrap','repeats','mmd_n','mmd_repeats','mmd_permutations',
                    'mmd_bootstrap','mmd_inference_n','mmd_median_n','shap_source_n',
                    'shap_error_n','window_size','monitor_windows']:
            if not isinstance(self.cfg[key],int) or self.cfg[key]<2:
                raise ValueError(f'{key} must be an integer >= 2')
        if self.cfg['threads']<1 or len(set(self.cfg['lofo_seeds']))<2:
            raise ValueError('Use at least one CPU thread and two distinct LOFO seeds')
        if len(set(self.cfg['lofo_seeds']))!=len(self.cfg['lofo_seeds']):
            raise ValueError('LOFO seeds must be distinct')
        if not all(0<f<=.5 for f in self.cfg['budget_fractions']):
            raise ValueError('Label fractions must be in (0,0.5], within the adaptation half')
        self.threadpool_controller=threadpoolctl.threadpool_limits(limits=self.cfg['threads'])
        import_names={'scikit-learn':'sklearn'}
        self.versions = {k: getattr(sys.modules[import_names.get(k,k)],'__version__',metadata.version(k))
                         for k in PACKAGES}
        self.distribution_versions = {k:metadata.version(k) for k in PACKAGES}
        mismatch = {k: (v, self.versions[k]) for k, v in PACKAGES.items()
                    if self.versions[k] != v}
        if mismatch and self.cfg['strict_versions']:
            raise RuntimeError(f'Install the pinned requirements then restart kernel: {mismatch}')
        stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
        self.out = Path(self.cfg['output_root']).expanduser().resolve() / f"{self.cfg['mode']}_{stamp}"
        for p in ['tables', 'figures', 'splits', 'models', 'predictions', 'features',
                  'shap', 'mmd', 'code']:
            (self.out / p).mkdir(parents=True, exist_ok=True)
        self.data, self.parts, self.models, self.scores = {}, {}, {}, {}
        self.provenance_files = {}
        self.tables, self.completed, self.input_files = {}, [], {}
        self.next_stage_index=0
        self.config_sha256=hashlib.sha256(json.dumps(self.cfg,sort_keys=True).encode()).hexdigest()
        self.manifest = {'status': 'running', 'mode': self.cfg['mode'],
                         'config': self.cfg, 'versions': self.versions,
                         'installed_distribution_versions':self.distribution_versions,
                         'reference_pin_mismatches':mismatch,
                         'installed_vs_loaded_mismatches':{
                             k:{'installed':self.distribution_versions[k],
                                'loaded':self.versions[k]}
                             for k in PACKAGES if self.distribution_versions[k]!=self.versions[k]},
                         'python': sys.version, 'platform': platform.platform(),
                         'started_utc': stamp, 'interval_level': 0.95}
        self.write_json('run_manifest.json', self.manifest)
        self.write_json('code/config.json', self.cfg)
        # E4/M6: A managed notebook kernel may preload NumPy/Matplotlib before
        # the first user cell. Installing a different distribution in that same
        # process cannot replace the modules already in memory. Reproduce the
        # versions that actually computed this run, while separately retaining
        # the reference pins and installed-distribution versions in the manifest.
        (self.out / 'code/requirements.txt').write_text(
            '\n'.join(f'{k}=={self.versions[k]}' for k in PACKAGES) + '\n')
        (self.out / 'code/reference_pins.txt').write_text(
            '\n'.join(f'{k}=={v}' for k, v in PACKAGES.items()) + '\n')
        self.write_json('code/environment_packages.json',
            sorted([{'name':d.metadata.get('Name','unknown'),'version':d.version}
                    for d in metadata.distributions()],key=lambda d:d['name'].lower()))
        plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 11,
                             'axes.spines.top': False, 'axes.spines.right': False})
        print('Run directory:', self.out, flush=True)

    def write_json(self, name, obj):
        p = self.out / name
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(json.dumps(jsonable(obj), indent=2, allow_nan=False) + '\n')

    def table(self, name, rows):
        df = rows if isinstance(rows, pd.DataFrame) else pd.DataFrame(rows)
        self.tables[name] = df
        df.to_csv(self.out / 'tables' / f'{name}.csv', index=False)
        return df

    def save_predictions(self, name, ids, labels, **scores):
        np.savez_compressed(self.out / 'predictions' / f'{name}.npz',
                            row_ids=np.asarray(ids, str), labels=np.asarray(labels, np.int8),
                            **{k: np.asarray(v) for k, v in scores.items()})

    def done(self, stage):
        self.completed.append(stage)
        self.manifest['completed_stages'] = self.completed
        self.write_json('run_manifest.json', self.manifest)
        print(f'Completed: {stage}', flush=True)

def execute_stage(S,function):
    # FIX A08: A failed/completed run is immutable. Correct CFG and initialize a
    # new RunState, rather than mixing refreshed models with stale downstream fits.
    if S.manifest['status'] in ('failed','complete'):
        raise RuntimeError('This run is failed or complete. Initialize a new RunState and Run all.')
    expected=STAGE_ORDER[S.next_stage_index] if S.next_stage_index<len(STAGE_ORDER) else 'no remaining stages'
    if function.__name__!=expected:
        raise RuntimeError(f'Execute stages in order; next stage is {expected}.')
    try:
        if hashlib.sha256(json.dumps(S.cfg,sort_keys=True).encode()).hexdigest()!=S.config_sha256:
            raise RuntimeError('Configuration changed during a run. Initialize a new RunState.')
        # A digest recorded after reading a modified file would not identify the
        # data actually used. Recheck the frozen input snapshot before every stage.
        if function.__name__!='stage_inputs':
            for domain,spec in {**S.input_files,**S.provenance_files}.items():
                if digest_file(spec['path'])!=spec['sha256']:
                    raise RuntimeError(f'{domain} input or cohort record changed during this run. Start a new run.')
        result=function(S)
        S.next_stage_index+=1
        S.manifest['executed_functions']=STAGE_ORDER[:S.next_stage_index]
        S.write_json('run_manifest.json',S.manifest)
        return result
    except Exception as exc:
        S.manifest.update(status='failed',failed_stage=function.__name__,error=str(exc))
        S.write_json('run_manifest.json',S.manifest)
        raise

## 2. Paths and frozen preparation

The original Google Drive project path is retained. Full corrected unified CSVs
remain inputs. The approved preparation folder supplies the exact source split
and both target row selections. Changing its files requires a deliberately new
preparation and updated fingerprints, not a silent redraw of source partitions.


In [ ]:
from pathlib import Path
RUN_SYNTHETIC_SELF_TEST=False
# Drive mounting and PROJECT_ROOT are configured in the first code cell.
CFG=default_config()
CFG['data_dir']=str(PROJECT_ROOT/'data')
CFG['output_root']=str(PROJECT_ROOT/'results'/'revision')
# Overlap review / M6: consume the approved, fingerprinted preparation files.
CFG['preparation_dir']=str(PROJECT_ROOT/'results/revision/url_grouped_preparation_20261020T224241295695Z')
# The full manifest records loaded and installed versions separately. A
# managed kernel can preload an older NumPy/Matplotlib before any user cell;
# the numerical run must use those loaded versions consistently, without pip.
CFG['strict_versions']=False

if RUN_SYNTHETIC_SELF_TEST:
    _fake=make_smoke_inputs('/tmp/unified_feature_smoke_data')
    CFG=smoke_config(_fake,'/tmp/unified_feature_smoke_results')
    print('SYNTHETIC VALIDATION ONLY; exclude these values from the paper.')
print(json.dumps(CFG,indent=2))

## 3. Statistical definitions and safeguards

**E2/E3/E4:** score thresholds include the attainable reject-all rule;
the paper must separate fixed 0.5, source-selected, target-adapted and
target-label oracle settings. Intervals use row-level stratified resampling,
conditional on a trained model. Source-only thresholds may exceed an FPR
budget on a target and are reported as such. The notebook does not call a
target-label oracle a deployable result.

In [ ]:
def require_two_classes(y, context):
    a=np.asarray(y)
    if a.ndim!=1 or not len(a) or not np.isin(a,[0,1]).all() or set(np.unique(a))!={0,1}:
        raise ValueError(f'{context} requires a nonempty 1D vector containing only labels 0 and 1, both present')

def finite_vector(values,name,length=None):
    a=np.asarray(values,float)
    if a.ndim!=1 or not len(a) or (length is not None and len(a)!=length) or not np.isfinite(a).all():
        raise ValueError(f'{name} must be a finite nonempty 1D vector of the expected length')
    return a

def validated_weights(weights,y):
    w=np.ones(len(y)) if weights is None else finite_vector(weights,'weights',len(y))
    if (w<0).any() or any(np.sum(w[np.asarray(y)==k])<=0 for k in (0,1)):
        raise ValueError('Weights must be nonnegative and give both classes positive support')
    return w

def ci(values):
    a = np.asarray(values, float)
    if not np.isfinite(a).all():
        raise AssertionError('A confidence interval contains non-finite bootstrap values')
    return tuple(np.quantile(a, [0.025, 0.975]))

class StratifiedRowResampler:
    def __init__(self, y, groups):
        self.y = np.asarray(y)
        _, self.inv = np.unique(groups, return_inverse=True)
        membership = pd.DataFrame({'g':self.inv, 'y':self.y}).groupby('g').y
        if (membership.nunique() > 1).any():
            raise ValueError('Conflicting labels within a resampling group')
        group_y = membership.first().to_numpy()
        self.strata = [np.flatnonzero(group_y == label) for label in np.unique(group_y)]
        self.n = len(group_y)

    def weights(self, rng):
        counts = np.zeros(self.n, dtype=np.int32)
        for g in self.strata:
            counts[g] = rng.multinomial(len(g), np.full(len(g), 1/len(g)))
        return counts[self.inv].astype(float)

class MetricCache:
    """Cache score order/tie boundaries, so bootstrap AP needs no repeated sorting."""
    def __init__(self, y, probabilities, ranking=None, threshold=0.5, decisions=None):
        # FIX A07: Validate before casting; int([0.2,1.7]) silently becomes [0,1].
        require_two_classes(y,'metric evaluation')
        self.y = np.asarray(y, int)
        self.p = finite_vector(probabilities,'probabilities',len(self.y))
        if not ((self.p>=0)&(self.p<=1)).all() or np.isnan(threshold):
            raise ValueError('Probabilities must be in [0,1], and the threshold cannot be NaN')
        if decisions is not None:
            d=np.asarray(decisions)
            if d.shape!=self.y.shape or not np.isin(d,[0,1]).all():
                raise ValueError('decisions must be a matching binary vector')
        self.pred = self.p >= threshold if decisions is None else np.asarray(decisions, bool)
        score = self.p if ranking is None else finite_vector(ranking,'ranking',len(self.y))
        self.f1_num=(2*self.y*self.pred).astype(float)
        self.f1_den=(self.y+self.pred).astype(float)
        self.order = np.argsort(-score, kind='stable')
        ordered_score = score[self.order]
        self.ends = np.r_[np.flatnonzero(ordered_score[:-1] != ordered_score[1:]), len(score)-1]
        self.bins = np.minimum((self.p * 15).astype(int), 14)

    def values(self, weights=None):
        w = validated_weights(weights,self.y)
        y, pred = self.y, self.pred
        tp, fp = np.sum(w * y * pred), np.sum(w * (1-y) * pred)
        fn, tn = np.sum(w * y * ~pred), np.sum(w * (1-y) * ~pred)
        ordered_w = w[self.order]
        hits = np.cumsum(ordered_w * y[self.order])[self.ends]
        totals = np.cumsum(ordered_w)[self.ends]
        precision = np.divide(hits, totals, out=np.zeros_like(hits), where=totals > 0)
        ap = float(np.sum(precision * np.diff(np.r_[0., hits])) / np.sum(w*y))
        bin_n = np.bincount(self.bins, weights=w, minlength=15)
        bin_p = np.bincount(self.bins, weights=w*self.p, minlength=15)
        bin_y = np.bincount(self.bins, weights=w*y, minlength=15)
        return {'F1': float(2*tp/(2*tp+fp+fn)) if 2*tp+fp+fn else 0.,
                'AP': ap, 'Brier': float(np.sum(w*(self.p-y)**2)/w.sum()),
                'ECE': float(np.abs(bin_p-bin_y).sum()/bin_n.sum()),
                'TPR': float(tp/(tp+fn)), 'FPR': float(fp/(fp+tn)),
                'Accuracy': float((tp+tn)/w.sum()),
                'Precision': float(tp/(tp+fp)) if tp+fp else 0.,
                'Recall': float(tp/(tp+fn)),
                'positive_fraction': float((tp+fp)/w.sum()),
                'TN': float(tn), 'FP': float(fp), 'FN': float(fn), 'TP': float(tp)}

    def f1_only(self,weights):
        # FIX A11: LOFO needs paired F1 deltas, not AP/ECE on every seed draw.
        # Two cached dot products compute the identical F1 estimand without
        # repeated sorting/cumulative sums or unrelated calibration metrics.
        numerator=float(np.dot(weights,self.f1_num))
        denominator=float(np.dot(weights,self.f1_den))
        return numerator/denominator if denominator else 0.

def metrics_with_ci(y, p, groups, n_boot, seed, ranking=None, threshold=0.5, decisions=None):
    cache = MetricCache(y, p, ranking, threshold, decisions)
    point = cache.values()
    draws = {k: [] for k in ['F1', 'AP', 'Brier', 'ECE', 'TPR', 'FPR']}
    probability_cache = MetricCache(y,p,threshold=threshold,decisions=decisions) if ranking is not None else None
    if probability_cache is not None:
        point['AP_probability']=probability_cache.values()['AP']
        draws['AP_probability']=[]
    rng = np.random.default_rng(seed)
    resampler = StratifiedRowResampler(y, groups)
    for _ in range(n_boot):
        weights=resampler.weights(rng)
        m = cache.values(weights)
        if probability_cache is not None:
            m['AP_probability']=probability_cache.values(weights)['AP']
        for k in draws:
            draws[k].append(m[k])
    for k, vals in draws.items():
        point[k+'_ci_low'], point[k+'_ci_high'] = ci(vals)
    point.update(ci_resamples=n_boot, ci_method='class-stratified exact-URL cluster percentile bootstrap; row-weighted metrics',
                 ci_estimand='fixed trained model, fixed evaluation partition')
    return point

def verify_core_math():
    """Independent library/brute-force checks run before processing real data."""
    y=np.array([0,1,0,1,1,0]);p=np.array([0,.5,.5,1,1,.2]);w=np.array([2,1,3,2,1,1])
    m=MetricCache(y,p).values(w)
    assert np.isclose(m['AP'],average_precision_score(y,p,sample_weight=w))
    assert np.isclose(m['F1'],f1_score(y,p>=.5,sample_weight=w))
    assert constrained_point(np.array([0,1]),np.array([1.,1.]),.01)==(np.inf,0.,0.)
    H=stable_entropy(np.array([0.,.1,-.1,1728.,-1728.]))
    assert H[0]==1 and H[1]==H[2] and np.isfinite(H).all()
    assert binomial_ci(0,1000)[1]>0 and binomial_ci(1000,1000)[0]<1
    print('Core numerical checks passed.',flush=True)

def exact_roc(y, score, weights=None):
    """Only attainable >= threshold decisions, including the reject-all rule."""
    y = np.asarray(y)
    require_two_classes(y, 'ROC')
    score=finite_vector(score,'ROC score',len(y))
    w=validated_weights(weights,y)
    order = np.argsort(-score, kind='stable')
    ends = np.r_[np.flatnonzero(score[order][:-1] != score[order][1:]), len(y)-1]
    tp = np.r_[0., np.cumsum(w[order]*y[order])[ends]]
    fp = np.r_[0., np.cumsum(w[order]*(1-y[order]))[ends]]
    return np.r_[np.inf, score[order][ends]], fp/np.sum(w*(1-y)), tp/np.sum(w*y)

def constrained_point(y, score, budget, weights=None):
    if not np.isfinite(budget) or not 0<=budget<=1:
        raise ValueError('FPR budget must be finite and in [0,1]')
    t, fpr, tpr = exact_roc(y, score, weights)
    feasible = np.flatnonzero(fpr <= budget + 1e-14)
    best = feasible[np.argmax(tpr[feasible])]  # first tie = highest threshold
    return float(t[best]), float(fpr[best]), float(tpr[best])

def choose_f1_threshold(y, score):
    """F1 optimum on the calibration/validation partition only; highest-threshold tie."""
    t, fpr, tpr = exact_roc(y, score)
    n0,n1=int(np.sum(np.asarray(y)==0)),int(np.sum(np.asarray(y)==1))
    tp, fp, fn = tpr*n1, fpr*n0, (1-tpr)*n1
    f = np.divide(2*tp, 2*tp+fp+fn, out=np.zeros_like(tp), where=(2*tp+fp+fn)>0)
    return float(t[np.argmax(f)])

def score_rates(y, score, t, weights=None):
    y, pred = np.asarray(y), np.asarray(score) >= t
    w = np.ones(len(y)) if weights is None else np.asarray(weights)
    return (float(np.sum(w*(1-y)*pred)/np.sum(w*(1-y))),
            float(np.sum(w*y*pred)/np.sum(w*y)))

def binomial_ci(k, n):
    """Clopper-Pearson interval; nondegenerate at zero/all alarms."""
    if not isinstance(k,(int,np.integer)) or not isinstance(n,(int,np.integer)) or n<1 or not 0<=k<=n:
        raise ValueError('Binomial counts require integer 0 <= k <= n and n >= 1')
    return (0. if k == 0 else float(beta.ppf(.025, k, n-k+1)),
            1. if k == n else float(beta.ppf(.975, k+1, n-k)))

def stable_entropy(z):
    """Binary entropy in bits directly from a finite logistic score."""
    a = np.abs(np.asarray(z, float))
    assert np.isfinite(a).all()
    h=(np.logaddexp(0., -a) + a*expit(-a))/np.log(2.)
    # FIX A05: Near z=0 this formula can round to 1+2e-16, dropping rows
    # outside Figure 4's [0,1] bins. Clip only final round-off, never logits/p.
    if not np.isfinite(h).all() or (h < -32*np.finfo(float).eps).any() or (h > 1+32*np.finfo(float).eps).any():
        raise ArithmeticError('Entropy exceeds its theoretical bounds beyond round-off')
    return np.clip(h,0.,1.)

def disjoint_groups(df, a, b):
    if set(df.iloc[a].url_group) & set(df.iloc[b].url_group):
        raise AssertionError('Exact URL identity overlaps between partitions')

def group_folds(df, n_splits, seed):
    # E2/overlap audit: URL identities, rather than row positions, define folds.
    y, groups = df.label.to_numpy(), df.url_group.to_numpy()
    by_group = df.groupby('url_group').label
    if by_group.nunique().gt(1).any():
        raise ValueError('Conflicting labels within a URL group')
    if by_group.first().value_counts().min() < n_splits:
        raise ValueError('Not enough distinct URLs per class for the requested folds')
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = [test for _, test in splitter.split(np.zeros(len(y)), y, groups)]
    for i, ix in enumerate(folds):
        require_two_classes(y[ix], 'URL-grouped split')
        for other in folds[i+1:]:
            disjoint_groups(df, ix, other)
    return folds

def save_split(S, name, df, memberships):
    rows = []
    seen = set()
    seen_groups = set()
    for part, ix in memberships.items():
        if seen.intersection(np.asarray(ix).tolist()):
            raise AssertionError(f'Overlapping row split {name}')
        seen.update(np.asarray(ix).tolist())
        current_groups = set(df.iloc[ix].url_group)
        if seen_groups & current_groups:
            raise AssertionError(f'URL identity crosses saved partitions: {name}')
        seen_groups.update(current_groups)
        block = df.iloc[ix][['row_id', 'raw_csv_row', 'url_group', 'label']].copy()
        block['partition'] = part
        rows.append(block)
    pd.concat(rows, ignore_index=True).to_csv(S.out/'splits'/f'{name}.csv.gz', index=False)

def new_model(name, cfg, seed=None):
    if name not in ('LR','GB'):
        raise ValueError('Model name must be LR or GB')
    seed = cfg['seed'] if seed is None else seed
    if name == 'LR':
        return LogisticRegression(class_weight='balanced', max_iter=1000,
                                  solver='lbfgs', random_state=seed)
    return LGBMClassifier(n_estimators=100, learning_rate=.05, max_depth=5,
                          num_leaves=31, subsample=.8, subsample_freq=0,
                          colsample_bytree=.8, class_weight='balanced',
                          random_state=seed, n_jobs=cfg['threads'], verbosity=-1,
                          deterministic=True, force_col_wise=True)

def fit_checked(model, X, y, feature_names=None):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        kwargs={'feature_name':list(feature_names)} if isinstance(model,LGBMClassifier) and feature_names is not None else {}
        model.fit(X, y, **kwargs)
    if any(issubclass(w.category, ConvergenceWarning) for w in caught):
        raise RuntimeError('LR did not converge. Increase max_iter, rerun, and record the change.')
    return model

def margin(model, X):
    z = model.decision_function(X) if isinstance(model, LogisticRegression) else model.booster_.predict(X, raw_score=True, num_threads=model.get_params()['n_jobs'])
    z = np.asarray(z, float)
    assert np.isfinite(z).all()
    return z

def fit_calibrator(method, z, y):
    if method not in ('Platt','isotonic'):
        raise ValueError('Calibrator must be Platt or isotonic')
    require_two_classes(np.asarray(y), 'calibrator fitting')
    z=finite_vector(z,'calibration margins',len(y))
    if method == 'Platt':
        fit = LogisticRegression(C=1e6, max_iter=5000, solver='lbfgs')
        fit_checked(fit, np.asarray(z).reshape(-1, 1), y)
        return fit
    return IsotonicRegression(out_of_bounds='clip').fit(z, y)

def calibrate(fit, z):
    return (fit.predict_proba(np.asarray(z).reshape(-1, 1))[:, 1]
            if isinstance(fit, LogisticRegression) else fit.predict(z))

def calibrated_logit(fit, z):
    if not isinstance(fit, LogisticRegression):
        raise TypeError('Stable logit entropy requires a Platt calibrator')
    return fit.coef_[0, 0]*np.asarray(z) + fit.intercept_[0]

def calibrator_description(fit):
    if isinstance(fit, LogisticRegression):
        slope = float(fit.coef_[0, 0])
        return {'method': 'Platt', 'C': 1e6, 'input': 'raw model margin',
                'slope': slope, 'intercept': float(fit.intercept_[0]),
                'rank_direction': 'increasing' if slope > 0 else 'decreasing' if slope < 0 else 'constant'}
    return {'method': 'isotonic', 'input': 'raw model margin',
            'x_thresholds': fit.X_thresholds_, 'y_thresholds': fit.y_thresholds_}

In [ ]:
verify_core_math()

## 4. Input identities and source training

E1/E2/M6: validate all full feature rows, raw labels, URL lengths, file hashes,
prepared indices and exact URL groups. Then apply the two target selections.
Source training/validation/test identities cannot overlap. The preprocessor and
classifiers fit source training only; calibration uses source validation.


In [ ]:
def source_grouped_parts(df, seed):
    """Predeclared first folds; URL groups never cross source partitions."""
    y, groups = df.label.to_numpy(), df.url_group.to_numpy()
    outer = StratifiedGroupKFold(5, shuffle=True, random_state=seed)
    development, test = next(outer.split(np.zeros(len(df)), y, groups))
    inner = StratifiedGroupKFold(5, shuffle=True, random_state=seed)
    tr, va = next(inner.split(np.zeros(len(development)), y[development], groups[development]))
    return {'source_train': np.sort(development[tr]),
            'source_validation': np.sort(development[va]), 'source_test': np.sort(test)}

def load_prepared_identity(S):
    """Overlap review/E1/E2/M6: verify frozen selections against corrected raw URLs.

    Raw/unified row alignment is checked through labels and URL lengths, together
    with approved file digests. These checks do not independently attest every
    upstream feature definition. Feature extraction remains a separate workflow.
    """
    if S.cfg['mode'] == 'smoke':
        # Synthetic input has no raw URLs; its unique row IDs are test-only units.
        S.source_indices = source_grouped_parts(S.data[SOURCE], S.cfg['seed'])
        S.manifest['identity_limit'] = 'Synthetic identities; not evidence about real URLs.'
        return
    root = Path(S.cfg['preparation_dir']).expanduser()
    for filename, expected in S.cfg['preparation_sha256'].items():
        path = root / filename
        if not path.is_file() or digest_file(path) != expected:
            raise ValueError(f'Preparation file missing or changed: {path}')
        S.provenance_files['preparation_' + filename] = {
            'path': str(path.resolve()), 'sha256': expected,
                'recorded_path': str(name)}
        (S.out / 'splits' / filename).write_bytes(path.read_bytes())
    source_record = json.loads((root / 'source_partition_record.json').read_text())
    with np.load(root / 'source_partitions.npz', allow_pickle=False) as saved:
        S.source_indices = {name: saved[name].copy() for name in saved.files}
    if set(S.source_indices) != {'source_train', 'source_validation', 'source_test'}:
        raise ValueError('Unexpected source partition names')
    combined = np.concatenate(list(S.source_indices.values()))
    if combined.dtype.kind not in 'iu' or not np.array_equal(
            np.sort(combined), np.arange(len(S.data[SOURCE]))):
        raise ValueError('Source partitions must cover every row exactly once')
    if source_record['seed'] != S.cfg['seed']:
        raise ValueError('Configuration seed differs from prepared source split')
    if source_record['split_file_sha256'] != digest_file(root / 'source_partitions.npz'):
        raise ValueError('Source record does not identify the saved split')

    approved_inputs = {}
    def verify_inputs(record):
        for name, expected in record['input_sha256'].items():
            path = relocated_dataset_path(name)
            if not path.is_file() or digest_file(path) != expected:
                raise ValueError(f'Prepared input missing or changed: {path}')
            S.provenance_files['input_' + str(path)] = {
                'path': str(path.resolve()), 'sha256': expected}
            approved_inputs[str(path.resolve())] = expected

    def attach(domain, raw_path, url_column, label_column, invert=False):
        full = S.data[domain]
        full_spec = S.input_files[domain]
        if approved_inputs.get(full_spec['path']) != full_spec['sha256']:
            raise ValueError(f'{domain}: loaded feature file differs from approved preparation input')
        if str(Path(raw_path).resolve()) not in approved_inputs:
            raise ValueError(f'{domain}: raw path is not identified by preparation')
        raw = pd.read_csv(raw_path, usecols=[url_column, label_column], keep_default_na=False)
        native = pd.to_numeric(raw[label_column], errors='raise').to_numpy()
        if not np.isin(native, [0, 1]).all():
            raise ValueError(f'{domain}: missing/unexpected raw labels')
        y = 1-native if invert else native
        original = raw[url_column].astype(str)
        urls = original.str.strip()
        if (len(raw) != len(full) or urls.eq('').any() or
                not np.array_equal(y, full.label.to_numpy()) or
                not np.array_equal(original.str.len().to_numpy(), full.URLLength.to_numpy())):
            raise ValueError(f'{domain}: raw/unified alignment or URL check failed')
        if pd.Series(y).groupby(urls.to_numpy()).nunique().gt(1).any():
            raise ValueError(f'{domain}: conflicting labels for a URL')
        # Common hashes allow cross-dataset identity checks without exporting URLs.
        full['url_group'] = urls.map(lambda u: hashlib.sha256(u.encode('utf-8')).hexdigest()).to_numpy()
        return urls

    verify_inputs(source_record)
    data_root = Path(S.cfg['data_dir'])
    source_urls = attach(SOURCE, data_root/'phiusiil.csv', 'URL', 'label')
    for i, first in enumerate(S.source_indices):
        require_two_classes(S.data[SOURCE].iloc[S.source_indices[first]].label, first)
        for second in list(S.source_indices)[i+1:]:
            disjoint_groups(S.data[SOURCE], S.source_indices[first], S.source_indices[second])
    # The prepared arrays are authoritative; do not redraw them under a different
    # sklearn version. The recorded version identifies their original generator.
    S.manifest['source_partition_record'] = source_record
    source_label = pd.Series(S.data[SOURCE].label.to_numpy(), index=source_urls.to_numpy())
    source_label = source_label[~source_label.index.duplicated()]
    audits = []
    for domain, stem, raw_name, url_col, label_col, invert in [
            ('URL-Phish', 'urlphish', 'URL-Phish.csv', 'url', 'label', False),
            ('LegitPhish', 'legitphish', 'LegitPhish Dataset.csv', 'URL', 'ClassLabel', True)]:
        record = json.loads((root / f'{stem}_selection_record.json').read_text())
        verify_inputs(record)
        if record['source_split_sha256'] != source_record['split_file_sha256']:
            raise ValueError(f'{domain}: selection uses a different source split')
        if record['selection_file_sha256'] != digest_file(root / f'{stem}_selection.npz'):
            raise ValueError(f'{domain}: selection record checksum differs')
        urls = attach(domain, data_root/raw_name, url_col, label_col, invert)
        mapped = urls.map(source_label)
        if (mapped.notna() & mapped.ne(S.data[domain].label)).any():
            raise ValueError(f'{domain}: conflicting cross-source URL labels')
        hits = {name: urls.isin(source_urls.iloc[ix]).to_numpy()
                for name, ix in S.source_indices.items()}
        expected = np.flatnonzero(~hits['source_train'] & ~hits['source_validation'])
        with np.load(root / f'{stem}_selection.npz', allow_pickle=False) as saved:
            ix = saved['corrected_full_row_indices'].copy()
            if ix.dtype.kind not in 'iu' or not np.array_equal(ix, expected):
                raise ValueError(f'{domain}: selection differs from exact development exclusion')
            codes, _ = pd.factorize(urls.iloc[ix], sort=True)
            if not np.array_equal(saved['url_group'], codes):
                raise ValueError(f'{domain}: saved URL groups differ from raw URLs')
            if not np.array_equal(saved['source_test_match'], hits['source_test'][ix]):
                raise ValueError(f'{domain}: source-test flags differ from raw URLs')
        selected = S.data[domain].iloc[ix].copy().reset_index(drop=True)
        require_two_classes(selected.label, domain)
        if len(ix) != record['selected_rows'] or selected.url_group.nunique() != record['distinct_selected_URLs']:
            raise ValueError(f'{domain}: selected row/group totals differ from record')
        expected_counts = S.cfg['datasets'][domain]['selected_counts']
        if {str(k):int((selected.label == k).sum()) for k in [0,1]} != expected_counts:
            raise ValueError(f'{domain}: unexpected selected class counts')
        S.data[domain] = selected
        print(f'Selected {domain}: {len(selected):,} rows; '
              f'{selected.url_group.nunique():,} distinct URLs', flush=True)
        S.input_files[domain]['evaluation_rows'] = len(selected)
        S.input_files[domain]['evaluation_label_counts'] = expected_counts
        audit = {'dataset':domain, 'full_rows':len(urls), 'selected_rows':len(ix),
                 'distinct_selected_URLs':selected.url_group.nunique(),
                 'legitimate':int((selected.label==0).sum()), 'phishing':int((selected.label==1).sum()),
                 'excluded_train_matches':int(hits['source_train'].sum()),
                 'excluded_validation_matches':int(hits['source_validation'].sum()),
                 'retained_source_test_matches':int(hits['source_test'][ix].sum()),
                 'group_definition':'SHA256 of exact URL after surrounding whitespace stripping'}
        audits.append(audit)
        S.write_json(f'splits/{stem}_cohort_audit.json', audit)
    S.table('00b_target_cohort_audit', audits)
    S.manifest['target_cohort_audit'] = audits
    S.manifest['cohort_provenance_files'] = S.provenance_files
    S.manifest['identity_limit'] = ('Exact URLs grouped; targets exclude source training/validation URLs. '
        'Source-test overlap remains in classifier comparisons. Domains are not grouped.')
    # Recheck all inputs after reading; execute_stage repeats this before each stage.
    for spec in S.provenance_files.values():
        if digest_file(spec['path']) != spec['sha256']:
            raise RuntimeError('Prepared input changed while loading')

def independent_reference(S):
    """M1/M2 overlap correction: common reference excludes both target URL sets.

    This conditions the reference on absence from the targets and can change its
    class mix and feature distribution. It is a restricted descriptive reference,
    not an estimate of the entire source-test population. Exact URL separation
    does not establish domain independence.
    """
    full = S.data[SOURCE].iloc[S.parts['source_test'][1]].copy()
    target_groups = set().union(*(set(S.data[d].url_group) for d in TARGETS))
    reference = full.loc[~full.url_group.isin(target_groups)].drop_duplicates('url_group').reset_index(drop=True)
    require_two_classes(reference.label, 'restricted source reference')
    for d in TARGETS:
        if set(reference.url_group) & set(S.data[d].url_group):
            raise AssertionError('Reference still shares target URLs')
    return reference

def stage_inputs(S):
    summaries=[]
    for domain,spec in S.cfg['datasets'].items():
        path=Path(S.cfg['data_dir']).expanduser()/spec['path']
        if not path.is_file():
            raise FileNotFoundError(f'{domain}: unified input CSV missing: {path}')
        before=digest_file(path)
        # E1/R2-4: preserve literal TLD categories such as 'missing'; reject
        # unexpected columns instead of silently training on IDs or extra labels.
        with path.open(encoding='utf-8-sig',newline='') as handle:
            header=next(csv.reader(handle),None)
        if header != FEATURES+['label']:
            raise ValueError(f'{domain}: expected exactly 15 ordered features and label; got {header}')
        df=pd.read_csv(path,keep_default_na=False,low_memory=False,
                       dtype={'TLD':'string'})
        if digest_file(path)!=before:
            raise RuntimeError(f'{domain}: file changed during read')
        if df.empty or df.isna().any().any():
            raise ValueError(f'{domain}: empty rows or missing values')
        for c in NUMERIC+['label']:
            df[c]=pd.to_numeric(df[c],errors='raise')
        labels=df.label.to_numpy()
        require_two_classes(labels,domain)
        if not np.isfinite(df[NUMERIC].to_numpy(dtype=float)).all():
            raise ValueError(f'{domain}: non-finite numeric feature')
        if (df[NUMERIC]<0).any().any():
            raise ValueError(f'{domain}: negative lexical feature')
        if not df.TLD.astype(str).str.len().ge(1).all():
            raise ValueError(f'{domain}: blank categorical TLD')
        if not np.allclose(df.LetterRatioInURL,
             np.divide(df.NoOfLettersInURL,df.URLLength,
                       out=np.zeros(len(df)),where=df.URLLength.to_numpy()!=0),
             rtol=1e-7,atol=1e-7):
            raise ValueError(f'{domain}: letter ratio/count invariant failed')
        if not np.allclose(df.DegitRatioInURL,
             np.divide(df.NoOfDegitsInURL,df.URLLength,
                       out=np.zeros(len(df)),where=df.URLLength.to_numpy()!=0),
             rtol=1e-7,atol=1e-7):
            raise ValueError(f'{domain}: digit ratio/count invariant failed')
        counts={str(v):int((labels==v).sum()) for v in (0,1)}
        if spec.get('expected_counts') is not None and counts!=spec['expected_counts']:
            raise ValueError(f'{domain}: harmonized label counts {counts} differ from configured {spec["expected_counts"]}')
        # Feature-only files have no URL/domain identifiers. These unique row
        # groups are bookkeeping, not proof of cross-dataset independence.
        df['row_id']=[f'{domain}:{i}' for i in range(len(df))]
        df['raw_csv_row']=np.arange(len(df))+2
        df['url_group']=df['row_id']
        df['label']=df['label'].astype(np.int8)
        S.data[domain]=df
        S.input_files[domain]={'path':str(path.resolve()),'sha256':before,
            'rows':len(df),'harmonized_label_counts':counts}
        summaries.append({'dataset':domain,'rows':len(df),
                          'legitimate':counts['0'],'phishing':counts['1'],
                          'phishing_fraction':counts['1']/len(df)})
        print(f'Validated {domain}: {len(df):,} rows, labels {counts}',flush=True)
    load_prepared_identity(S)
    summaries=[{'dataset':d, 'rows':len(a),
        'legitimate':int((a.label==0).sum()), 'phishing':int((a.label==1).sum()),
        'phishing_fraction':float(a.label.mean()), 'distinct_URLs':a.url_group.nunique()}
        for d,a in S.data.items()]
    S.table('00_label_and_extraction_sanity',summaries)
    S.manifest['input_files']=S.input_files
    if S.cfg['mode']=='smoke':
        S.manifest['identity_limit']='Synthetic row identity only; no URL/domain grouping.'
    S.done('validated_unified_feature_inputs')

def stage_splits_and_models(S):
    # E1/E2/overlap review: use the verified prepared URL-grouped source split.
    # The encoder and classifiers still fit source training features/labels only.
    df=S.data[SOURCE]; y=df.label.to_numpy()
    train=S.source_indices['source_train']
    val=S.source_indices['source_validation']
    test=S.source_indices['source_test']
    S.parts={'source_train':(SOURCE,train),
             'source_validation':(SOURCE,val),'source_test':(SOURCE,test),
             **{d:(d,np.arange(len(S.data[d]))) for d in TARGETS}}
    save_split(S,'source_train_validation_test',df,
               {'train':train,'validation':val,'test':test})
    S.table('01_dataset_splits',[{'partition':k,'dataset':d,'rows':len(ix),
        'legitimate':int((S.data[d].iloc[ix].label==0).sum()),
        'phishing':int((S.data[d].iloc[ix].label==1).sum()),
        'unique_url_groups':S.data[d].iloc[ix].url_group.nunique(),
        'group_definition':'exact URL after surrounding whitespace stripping; domains not grouped'}
        for k,(d,ix) in S.parts.items()])
    # M1/M2: save the whole restricted reference so its scope can be audited.
    reference=independent_reference(S)
    source_test=df.iloc[test]
    S.table('00c_restricted_reference_population',[{
        'source_test_rows':len(source_test),
        'source_test_distinct_URLs':source_test.url_group.nunique(),
        'reference_distinct_URLs':len(reference),
        'reference_legitimate':int((reference.label==0).sum()),
        'reference_phishing':int((reference.label==1).sum()),
        'scope':'source-test URLs absent from BOTH selected targets; one row per URL; not full source-test inference'}])
    save_split(S,'restricted_source_reference',reference,
               {'MMD_and_entropy_reference':np.arange(len(reference))})
    S.encoder=ColumnTransformer([
        ('numeric',StandardScaler(),NUMERIC),
        ('tld',OneHotEncoder(handle_unknown='ignore',sparse_output=True,
                            dtype=np.float64),['TLD'])],sparse_threshold=1.0)
    S.encoder.fit(df.iloc[train][FEATURES])
    S.X={k:sparse.csr_matrix(S.encoder.transform(S.data[d].iloc[ix][FEATURES]))
         for k,(d,ix) in S.parts.items()}
    vocab=S.encoder.named_transformers_['tld'].categories_[0]
    S.feature_raw=np.asarray(NUMERIC+['TLD']*len(vocab))
    S.feature_labels=np.asarray(NUMERIC+[f'TLD={v}' for v in vocab])
    S.feature_safe=[f'f{i:05d}' for i in range(len(S.feature_raw))]
    S.masks={'Length':np.isin(S.feature_raw,LENGTH),
             'Count':np.isin(S.feature_raw,COUNT),
             'DomainProtocol':np.isin(S.feature_raw,DOMAIN)|(S.feature_raw=='TLD'),
             'TLD':S.feature_raw=='TLD','HTTPS':S.feature_raw=='IsHTTPS'}
    assert (S.masks['Length']|S.masks['Count']|S.masks['DomainProtocol']).all()
    S.table('S2_fitted_TLD_vocabulary',pd.DataFrame({'TLD':vocab}))
    S.table('S2_encoded_feature_mapping',pd.DataFrame({'position':range(len(vocab)+14),
        'safe_name':S.feature_safe,'raw_feature':S.feature_raw,
        'display_name':S.feature_labels}))
    S.table('04_feature_family_composition',[{'family':k,
        'removed_columns':int(m.sum()),'retained_columns':int((~m).sum()),
        'total_encoded_columns':len(m),
        'overlap_note':'Intentional: DomainLength/TLDLength in Length and Domain; NoOfSubDomain in Count and Domain'}
        for k,m in S.masks.items()])
    S.write_json('models/feature_masks.json',
                 {k:np.flatnonzero(m) for k,m in S.masks.items()})
    S.table('02_unknown_TLD_rates',[{'dataset':d,
        'unknown_rows':int((~a.TLD.isin(vocab)).sum()),'rows':len(a),
        'unknown_fraction':float((~a.TLD.isin(vocab)).mean()),
        'unknown_categories':int(a.loc[~a.TLD.isin(vocab),'TLD'].nunique())}
        for d,a in S.data.items()])
    joblib.dump(S.encoder,S.out/'models/source_only_preprocessor.joblib')
    ytrain=df.iloc[train].label.to_numpy()
    S.source_calibrators,S.source_thresholds={},{}
    for name in ['LR','GB']:
        model=fit_checked(new_model(name,S.cfg),S.X['source_train'],ytrain,
                          S.feature_safe)
        S.models[name]=model
        joblib.dump(model,S.out/'models'/f'{name}_baseline.joblib')
        S.write_json(f'models/{name}_parameters.json',model.get_params())
        if name=='GB':
            assert model.booster_.feature_name()==S.feature_safe
            model.booster_.save_model(str(S.out/'models/GB_baseline.txt'))
            S.write_json('models/GB_tree_dump.json',model.booster_.dump_model())
        else:
            S.table('S1_LR_coefficients',pd.DataFrame({
                'feature':S.feature_labels,'coefficient':model.coef_[0]}))
        S.scores[name]={}
        for part,(d,ix) in S.parts.items():
            z=margin(model,S.X[part]); p=expit(z)
            S.scores[name][part]=(z,p)
            block=S.data[d].iloc[ix]
            S.save_predictions(f'baseline_{name}_{part}',block.row_id,
                               block.label,margin=z,probability=p)
        va=df.iloc[val]; zv=S.scores[name]['source_validation'][0]
        S.source_thresholds[name,'raw_margin']=choose_f1_threshold(va.label.to_numpy(),zv)
        S.source_thresholds[name,'raw_probability']=choose_f1_threshold(
            va.label.to_numpy(),S.scores[name]['source_validation'][1])
        for method in ['Platt','isotonic']:
            fit=fit_calibrator(method,zv,va.label.to_numpy())
            S.source_calibrators[name,method]=fit
            joblib.dump(fit,S.out/'models'/f'{name}_source_validation_{method}.joblib')
            S.write_json(f'models/{name}_source_validation_{method}.json',
                         calibrator_description(fit))
            S.source_thresholds[name,method]=choose_f1_threshold(
                va.label.to_numpy(),calibrate(fit,zv))
    S.write_json('models/source_F1_thresholds.json',
        {f'{n}_{m}':v for (n,m),v in S.source_thresholds.items()})
    S.done('source_splits_encoder_and_models')

In [ ]:
# M6: record the stage sources actually executed by this notebook.
# Run all from a fresh kernel so these defined functions match the digest.
# Source code is the published notebook. Input file digests and the
# run configuration are recorded in run_manifest.json.
S=RunState(CFG)
S.write_json('run_manifest.json',S.manifest)

In [ ]:
execute_stage(S, stage_inputs)

In [ ]:
execute_stage(S, stage_splits_and_models)

In [ ]:
display(S.tables["00_label_and_extraction_sanity"],S.tables["01_dataset_splits"],
        S.tables["04_feature_family_composition"],S.tables["02_unknown_TLD_rates"])

## 5. Zero-shot transfer, operating points and saturation

**E3/R1-3:** evaluate both models on source test and both external datasets.
ROC rules use raw decision scores, attained thresholds, and achieved FPR/TPR
at illustrative 0.1%, 1% and 5% ceilings. Source thresholds come from
source validation and transfer unchanged. Target-label thresholds are labeled
oracle diagnostics. Save exact threshold hex, score vectors, saturation rates,
resampling intervals and the complete table for each model/domain/budget.

In [ ]:
def stage_baseline_and_operating_points(S):
    zero, saturation, operating = [], [], []
    rng = np.random.default_rng(S.cfg['seed']+300)
    va = S.data[SOURCE].iloc[S.parts['source_validation'][1]]
    va_resampler = StratifiedRowResampler(va.label, va.url_group)
    for model in ['LR','GB']:
        for part in ['source_test']+TARGETS:
            d, ix = S.parts[part]
            block = S.data[d].iloc[ix]
            y, groups = block.label.to_numpy(), block.url_group.to_numpy()
            z, p = S.scores[model][part]
            row = {'model':model, 'dataset':d, 'threshold':.5,
                   'threshold_provenance':'fixed raw-probability 0.5',
                   **metrics_with_ci(y,p,groups,S.cfg['bootstrap'],S.cfg['seed']+100,
                                     ranking=z),
                   'AP_probability':float(average_precision_score(y,p)),
                   'AUROC_margin':float(roc_auc_score(y,z))}
            row['AP_margin'] = row['AP']
            row['AP_margin_ci_low'],row['AP_margin_ci_high']=row['AP_ci_low'],row['AP_ci_high']
            zero.append(row)
            for label in ['all',0,1]:
                take = np.ones(len(y),bool) if label=='all' else y==label
                zz, pp = z[take], p[take]
                saturation.append({'model':model,'dataset':d,'class':label,'rows':int(take.sum()),
                    'probability_equals_0':float(np.mean(pp==0)),
                    'probability_equals_1':float(np.mean(pp==1)),
                    'probability_ge_0999':float(np.mean(pp>=.999)),
                    'probability_le_0001':float(np.mean(pp<=.001)),
                    'unique_margins':len(np.unique(zz)), 'unique_probabilities':len(np.unique(pp)),
                    'min_margin':float(zz.min()),'max_margin':float(zz.max()),
                    **{f'margin_q{int(q*100):02d}':float(np.quantile(zz,q)) for q in [.01,.5,.99]}})
            eval_resampler = StratifiedRowResampler(y,groups)
            zva = S.scores[model]['source_validation'][0]
            # Cache attainable thresholds; sorting is reused across bootstrap draws.
            va_order = np.argsort(-zva,kind='stable')
            va_ends = np.r_[np.flatnonzero(np.diff(zva[va_order])!=0),len(zva)-1]
            ev_order = np.argsort(-z,kind='stable')
            ev_ends = np.r_[np.flatnonzero(np.diff(z[ev_order])!=0),len(z)-1]
            def cached_point(yy, zz, w, order, ends, budget):
                tp = np.r_[0.,np.cumsum(w[order]*yy[order])[ends]]
                fp = np.r_[0.,np.cumsum(w[order]*(1-yy[order]))[ends]]
                tpr, fpr = tp/np.sum(w*yy),fp/np.sum(w*(1-yy))
                feasible = np.flatnonzero(fpr<=budget+1e-14)
                j = feasible[np.argmax(tpr[feasible])]
                return float(np.r_[np.inf,zz[order][ends]][j]),float(fpr[j]),float(tpr[j])
            policies = {}
            for budget in S.cfg['fpr_budgets']:
                tsource,_,_ = constrained_point(va.label.to_numpy(),zva,budget)
                toracle,_,_ = constrained_point(y,z,budget)
                policies[budget] = (tsource,toracle)
            draws = {(b,k):{'TPR':[],'FPR':[]} for b in policies for k in ['source_validation_policy','evaluation_label_oracle']}
            for _ in range(S.cfg['bootstrap']):
                wva, wev = va_resampler.weights(rng),eval_resampler.weights(rng)
                for budget in policies:
                    tt,_,_ = cached_point(va.label.to_numpy(),zva,wva,va_order,va_ends,budget)
                    f,t = score_rates(y,z,tt,wev)
                    draws[budget,'source_validation_policy']['FPR'].append(f)
                    draws[budget,'source_validation_policy']['TPR'].append(t)
                    _,f,t = cached_point(y,z,wev,ev_order,ev_ends,budget)
                    draws[budget,'evaluation_label_oracle']['FPR'].append(f)
                    draws[budget,'evaluation_label_oracle']['TPR'].append(t)
            for budget, thresholds in policies.items():
                for policy, threshold in zip(['source_validation_policy','evaluation_label_oracle'],thresholds):
                    fpr,tpr = score_rates(y,z,threshold)
                    row = {'model':model,'dataset':d,'FPR_budget':budget,'policy':policy,
                        'threshold_margin':threshold,
                        # FIX A13: A CSV parser can shift a decimal threshold by
                        # one ULP and flip all observations exactly at its score.
                        # Hex preserves the binary float; export decisions too.
                        'threshold_margin_hex':float(threshold).hex(),
                        'achieved_FPR':fpr,'TPR':tpr,
                        'budget_met_on_evaluation':fpr<=budget+1e-14,
                        'reject_all':bool(np.isinf(threshold)),
                        'threshold_provenance':'source validation only' if policy.startswith('source') else 'evaluation labels: oracle diagnostic, not deployable',
                        'CI_resamples':S.cfg['bootstrap'],
                        'CI_method':'class-stratified exact-URL cluster bootstrap with threshold reselection on its designated partition'}
                    for k in ['FPR','TPR']:
                        row[k+'_ci_low'],row[k+'_ci_high'] = ci(draws[budget,policy][k])
                    operating.append(row)
                    S.save_predictions(f'operating_{model}_{part}_{repr(float(budget))}_{policy}',
                        block.row_id,y,margin=z,decision=(z>=threshold).astype(np.int8))
    S.table('05_06_zero_shot_with_CI',zero)
    S.table('18_operating_points_all_domains',operating)
    S.table('19_probability_saturation',saturation)
    # Exact LR decomposition identifies score contributors without a causal claim.
    decomposition = []
    coef = S.models['LR'].coef_[0]
    for part in ['source_test']+TARGETS:
        d,ix = S.parts[part]
        y = S.data[d].iloc[ix].label.to_numpy()
        z,p = S.scores['LR'][part]
        contrib = S.X[part].multiply(coef).tocsr()
        residual = np.asarray(contrib.sum(axis=1)).ravel()+S.models['LR'].intercept_[0]-z
        assert np.max(np.abs(residual))<1e-7
        for label in [0,1]:
            for group,take in [('all',y==label),('probability_equals_1',(y==label)&(p==1)),
                               ('probability_below_0999',(y==label)&(p<.999))]:
                if not take.any():
                    continue
                cm = np.asarray(contrib[take,:14].mean(axis=0)).ravel()
                for j,f in enumerate(NUMERIC):
                    decomposition.append({'dataset':d,'class':label,'score_group':group,
                        'feature':f,'rows':int(take.sum()),'mean_margin_contribution':float(cm[j]),
                        'interpretation':'exact additive LR contribution; association, not causal effect'})
                decomposition.append({'dataset':d,'class':label,'score_group':group,
                    'feature':'all TLD indicators','rows':int(take.sum()),
                    'mean_margin_contribution':float(contrib[take,14:].sum()/take.sum()),
                    'interpretation':'exact additive LR contribution; association, not causal effect'})
    S.table('S1_LR_saturation_contributors',decomposition)
    S.done('baseline_CIs_saturation_and_operating_points')

In [ ]:
execute_stage(S, stage_baseline_and_operating_points)

In [ ]:
display(S.tables["05_06_zero_shot_with_CI"],S.tables["18_operating_points_all_domains"])

## 6. URL-disjoint recalibration and distinct-URL label budgets

E2/R1-2: target halves keep all copies of each URL together. Budgets acquire
one representative per distinct URL and use distinct selected URLs as their
denominator. Evaluation retains its rows; repeated-draw summaries are descriptive.
AP ranks raw model margins consistently; AP_probability exposes transformed-score
ties. Brier/ECE continue to use each variant's probabilities.


In [ ]:
def target_halves(df, seed):
    folds = group_folds(df,2,seed)
    adapt, test = folds[0],folds[1]
    disjoint_groups(df,adapt,test)
    return adapt,test

def summarize_repeats(df, keys, value_columns):
    rows=[]
    for key,block in df.groupby(keys,dropna=False,sort=False):
        key = key if isinstance(key,tuple) else (key,)
        row=dict(zip(keys,key)); row['draws']=len(block)
        for col in value_columns:
            values=block[col].to_numpy(float)
            row[col+'_mean']=float(values.mean())
            row[col+'_SD']=float(values.std(ddof=1)) if len(values)>1 else 0.
            row[col+'_repeat_p025'],row[col+'_repeat_p975']=ci(values)
        row['AP_definition']='raw model margin; AP_probability ranks probabilities with ties retained'
        row['interval_note']='repeat percentiles describe split/draw variability; SD and percentiles are not a CI of the mean'
        rows.append(row)
    return pd.DataFrame(rows)

def stage_calibration(S):
    rows, interval_rows, rank_rows = [],[],[]
    for domain in TARGETS:
        df = S.data[domain]
        y = df.label.to_numpy()
        for repeat in range(S.cfg['repeats']):
            adapt,test = target_halves(df,S.cfg['seed']+1000+repeat)
            save_split(S,f'calibration_{domain}_{repeat:02d}',df,{'adaptation':adapt,'evaluation':test})
            for model in ['LR','GB']:
                z,p = S.scores[model][domain]
                variants=[('raw_fixed_0.5',p,.5,p,'fixed 0.5',None),
                    ('raw_source_margin_threshold',p,S.source_thresholds[model,'raw_margin'],z,'source validation',None),
                    ('raw_source_probability_threshold',p,S.source_thresholds[model,'raw_probability'],p,'source validation',None),
                    ('raw_probability_threshold',p,choose_f1_threshold(y[adapt],p[adapt]),p,'target adaptation',None),
                    ('raw_margin_threshold',p,choose_f1_threshold(y[adapt],z[adapt]),z,'target adaptation',None)]
                for method in ['Platt','isotonic']:
                    source_fit=S.source_calibrators[model,method]
                    source_p=calibrate(source_fit,z)
                    variants.extend([(f'source_{method}_fixed_0.5',source_p,.5,source_p,'fixed 0.5',source_fit),
                        (f'source_{method}_source_threshold',source_p,S.source_thresholds[model,method],source_p,'source validation',source_fit)])
                    fit=fit_calibrator(method,z[adapt],y[adapt])
                    pp=calibrate(fit,z)
                    joblib.dump(fit,S.out/'models'/f'{domain}_{model}_{method}_repeat{repeat:02d}.joblib')
                    S.write_json(f'models/{domain}_{model}_{method}_repeat{repeat:02d}.json',calibrator_description(fit))
                    variants.extend([(f'target_{method}_fixed_0.5',pp,.5,pp,'fixed 0.5',fit),
                        (f'target_{method}_adaptation_threshold',pp,choose_f1_threshold(y[adapt],pp[adapt]),pp,'target adaptation',fit)])
                # Explicit test-label oracle is a diagnostic upper bound only.
                variants.append(('evaluation_margin_oracle',p,choose_f1_threshold(y[test],z[test]),z,'evaluation labels: oracle diagnostic',None))
                pred_export={'margin':z[test],'raw_probability':p[test]}
                for variant,pp,t,decision_score,provenance,fit in variants:
                    decisions=decision_score[test]>=t
                    metrics=MetricCache(y[test],pp[test],ranking=z[test],decisions=decisions).values()
                    metrics['AP_probability']=float(average_precision_score(y[test],pp[test]))
                    row={'dataset':domain,'model':model,'repeat':repeat,'variant':variant,
                         'adaptation_rows':len(adapt),'evaluation_rows':len(test),
                         'threshold':t,'threshold_hex':float(t).hex(),
                         'threshold_provenance':provenance,
                         'threshold_space':'raw margin' if variant in ('raw_margin_threshold','raw_source_margin_threshold','evaluation_margin_oracle') else 'probability',
                         'AP_ranking':'raw model margin; AP_probability records transformed-score ties',
                         'calibrator_fit_partition': 'target adaptation' if variant.startswith('target_') else 'source validation' if variant.startswith('source_') else 'none',
                         **metrics,'AP_raw_margin':float(average_precision_score(y[test],z[test])),
                         'unique_calibrated_scores':len(np.unique(pp[test]))}
                    if fit is not None and isinstance(fit,LogisticRegression):
                        row['Platt_slope']=float(fit.coef_[0,0])
                    rows.append(row)
                    pred_export[variant+'_probability']=pp[test]
                    pred_export[variant+'_decision']=decisions.astype(np.int8)
                    if repeat==0:
                        # CI is for this prespecified split, not for the mean of 20 splits.
                        ir={'dataset':domain,'model':model,'variant':variant,'repeat':0,
                            'threshold':t,'threshold_hex':float(t).hex(),
                            'threshold_provenance':provenance,
                            'threshold_space':row['threshold_space'],'AP_ranking':row['AP_ranking'],
                            **metrics_with_ci(y[test],pp[test],df.iloc[test].url_group.to_numpy(),
                                S.cfg['bootstrap'],S.cfg['seed']+2000,ranking=z[test],decisions=decisions)}
                        if 'oracle' in variant:
                            ir['ci_estimand']='fixed evaluation-label oracle threshold, conditional diagnostic only'
                        interval_rows.append(ir)
                        rank_rows.append({'dataset':domain,'model':model,'variant':variant,
                            'AP_margin':float(average_precision_score(y[test],z[test])),
                            'AP_raw_probability':float(average_precision_score(y[test],p[test])),
                            'AP_calibrated_probability':metrics['AP_probability'],
                            'unique_margins':len(np.unique(z[test])),
                            'unique_raw_probabilities':len(np.unique(p[test])),
                            'unique_calibrated_probabilities':len(np.unique(pp[test])),
                            'ranking_note':'isotonic ties and finite-precision sigmoid saturation can change AP; Platt slope is recorded'})
                S.save_predictions(f'calibration_{domain}_{model}_{repeat:02d}',df.iloc[test].row_id,y[test],**pred_export)
            print(f'Calibration {domain}: {repeat+1}/{S.cfg["repeats"]}',flush=True)
    allrows=S.table('12_recalibration_all_repeats',rows)
    S.table('12_recalibration_mean_SD',summarize_repeats(allrows,['dataset','model','variant'],['F1','AP','AP_probability','Brier','ECE']))
    S.table('12_recalibration_prespecified_split_CI',interval_rows)
    S.table('E2_AP_margin_probability_and_calibrated',rank_rows)
    S.done('disjoint_calibration_both_targets')

def stratified_budget(pool, y, n, seed, groups):
    """R1-2/E2: acquire n distinct URL labels, one representative row per URL.

    Repeated copies do not consume extra labels or extra calibrator-fitting
    weight. Selection is simulated label-informed stratification, not a claim
    that labels are known during real-world acquisition.
    """
    pool = np.asarray(pool, dtype=np.int64)
    frame = pd.DataFrame({'ix':pool, 'g':np.asarray(groups)[pool], 'y':np.asarray(y)[pool]})
    if frame.groupby('g').y.nunique().gt(1).any():
        raise ValueError('A label-budget URL has conflicting labels')
    unique = frame.drop_duplicates('g').ix.to_numpy()
    if n < 2 or n > len(unique):
        raise ValueError(f'Budget {n} must be between 2 and {len(unique)} unique pool URLs')
    rng = np.random.default_rng(seed)
    pools = [unique[np.asarray(y)[unique] == k] for k in [0,1]]
    if any(len(p) == 0 for p in pools):
        raise ValueError('Adaptation pool must contain both classes')
    n1 = min(max(1, int(round(n*len(pools[1])/len(unique)))), len(pools[1]), n-1)
    n0 = n-n1
    if n0 > len(pools[0]):
        n0 = len(pools[0]); n1 = n-n0
    ix = np.r_[rng.choice(pools[0], n0, False), rng.choice(pools[1], n1, False)]
    rng.shuffle(ix)
    require_two_classes(np.asarray(y)[ix], 'unique URL label budget')
    return ix

def stage_label_budgets(S):
    rows,intervals=[] ,[]
    for domain in TARGETS:
        df=S.data[domain]; y=df.label.to_numpy()
        for repeat in range(S.cfg['repeats']):
            pool,test=target_halves(df,S.cfg['seed']+3000+repeat)
            save_split(S,f'budget_pool_{domain}_{repeat:02d}',df,{'adaptation_pool':pool,'untouched_evaluation':test})
            # Zero-target-label controls use the identical evaluation partition.
            for model in ['LR','GB']:
                z,p=S.scores[model][domain]
                cp=calibrate(S.source_calibrators[model,'Platt'],z[test])
                controls=[('raw_fixed_0.5',p[test],.5),
                          ('source_Platt_source_threshold',cp,S.source_thresholds[model,'Platt'])]
                prediction={}
                for method,prob,t in controls:
                    rows.append({'dataset':domain,'model':model,'method':method,'budget_fraction':0.,
                        'labeled_rows':0,'labeled_url_groups':0,'repeat':repeat,'evaluation_rows':len(test),
                        'achieved_budget_fraction':0.,'labeled_legitimate':0,'labeled_phishing':0,
                        'threshold':t,'threshold_hex':float(t).hex(),
                        'threshold_provenance':'zero target labels; fixed 0.5 or source validation',
                        'AP_probability':float(average_precision_score(y[test],prob)),
                        **MetricCache(y[test],prob,ranking=z[test],threshold=t).values()})
                    prediction[method+'_probability']=prob
                    prediction[method+'_decision']=(prob>=t).astype(np.int8)
                    if repeat==0:
                        intervals.append({'dataset':domain,'model':model,'method':method,'budget_fraction':0.,
                            'repeat':0,'threshold':t,'threshold_hex':float(t).hex(),
                            **metrics_with_ci(y[test],prob,df.iloc[test].url_group.to_numpy(),
                                S.cfg['bootstrap'],S.cfg['seed']+5000,ranking=z[test],threshold=t)})
                S.save_predictions(f'budget_zero_label_controls_{domain}_{model}_{repeat:02d}',df.iloc[test].row_id,y[test],**prediction)
            for fraction_index,fraction in enumerate(S.cfg['budget_fractions']):
                # FIX A10: Report rounding honestly; do not call a forced minimum
                # of two labels the exact requested tiny budget. Fail clearly if
                # the URL-disjoint adaptation pool cannot supply that budget.
                n=int(round(fraction*df.url_group.nunique()))
                if n<2 or n>df.iloc[pool].url_group.nunique():
                    raise ValueError(f'{domain}: fraction {fraction} rounds to {n} labels, but a two-class fit needs 2..{df.iloc[pool].url_group.nunique()} unique URLs. Choose a feasible fraction and start a new run.')
                token=repr(float(fraction))  # round-trip unique; .3f collided for nearby fractions
                adapt=stratified_budget(pool,y,n,S.cfg['seed']+4000+repeat+fraction_index*100003,df.url_group.to_numpy())
                disjoint_groups(df,adapt,test)
                save_split(S,f'budget_{domain}_{token}_{repeat:02d}',df,{'labeled_adaptation':adapt,'untouched_evaluation':test})
                for model in ['LR','GB']:
                    z,_=S.scores[model][domain]
                    pred={}
                    for method in ['Platt','isotonic']:
                        fit=fit_calibrator(method,z[adapt],y[adapt]);p=calibrate(fit,z[test])
                        t=choose_f1_threshold(y[adapt],calibrate(fit,z[adapt]))
                        row={'dataset':domain,'model':model,'method':method,'budget_fraction':fraction,
                             'labeled_rows':len(adapt),'labeled_url_groups':df.iloc[adapt].url_group.nunique(),
                             'achieved_budget_fraction':len(adapt)/df.url_group.nunique(),
                             'budget_denominator_unique_URLs':int(df.url_group.nunique()),
                             'budget_unit':'distinct URL label; one representative per acquired URL',
                             'labeled_legitimate':int(np.sum(y[adapt]==0)),
                             'labeled_phishing':int(np.sum(y[adapt]==1)),
                             'repeat':repeat,'evaluation_rows':len(test),'threshold':t,
                             'threshold_hex':float(t).hex(),
                             'threshold_provenance':'same labeled adaptation subset; evaluation untouched',
                             'AP_ranking':'raw model margin; AP_probability retains probability ties',
                             'AP_probability':float(average_precision_score(y[test],p)),
                             **MetricCache(y[test],p,ranking=z[test],threshold=t).values()}
                        rows.append(row)
                        stem=f'budget_{domain}_{model}_{method}_{token}_{repeat:02d}'
                        joblib.dump(fit,S.out/'models'/f'{stem}.joblib')
                        S.write_json(f'models/{stem}.json',calibrator_description(fit))
                        pred[method+'_probability']=p;pred[method+'_decision']=(p>=t).astype(np.int8)
                        if repeat==0:
                            intervals.append({'dataset':domain,'model':model,'method':method,
                                'budget_fraction':fraction,'repeat':0,
                                'labeled_rows':len(adapt),'achieved_budget_fraction':len(adapt)/df.url_group.nunique(),
                             'budget_denominator_unique_URLs':int(df.url_group.nunique()),
                             'budget_unit':'distinct URL label; one representative per acquired URL',
                                'threshold':t,'threshold_hex':float(t).hex(),
                                'threshold_space':'probability',
                                'threshold_provenance':'labeled adaptation subset only',
                                **metrics_with_ci(y[test],p,df.iloc[test].url_group.to_numpy(),
                                  S.cfg['bootstrap'],S.cfg['seed']+5000,ranking=z[test],threshold=t)})
                    S.save_predictions(f'budget_{domain}_{model}_{token}_{repeat:02d}',df.iloc[test].row_id,y[test],**pred)
            print(f'Label budgets {domain}: {repeat+1}/{S.cfg["repeats"]}',flush=True)
    allrows=S.table('21_label_budgets_all_draws',rows)
    S.table('21_label_budgets_mean_SD',summarize_repeats(allrows,['dataset','model','method','budget_fraction'],['F1','AP','AP_probability','Brier','ECE']))
    S.table('21_label_budgets_prespecified_draw_CI',intervals)
    S.done('repeated_label_budgets_both_targets')

In [ ]:
execute_stage(S, stage_calibration)

In [ ]:
execute_stage(S, stage_label_budgets)

In [ ]:
display(S.tables["12_recalibration_mean_SD"],S.tables["21_label_budgets_mean_SD"])

## 7. Drift, pruning and LOFO

**E1/E4/M4/R2-8:** within-dataset label AUC is distinguished from
source-versus-target drift. Numeric conditional KS and encoded marginal KS
are exported. Pruning by target-feature KS and choosing the best K are
**retrospective diagnostics**, not deployable source-only selection. LOFO
exports explicit overlapping masks, TLD-only/HTTPS-only checks, both target
results, source-test cost and paired deltas defined as ablated minus baseline.

In [ ]:
def stage_shift_and_pruning(S):
    source_ix=S.parts['source_train'][1]
    source=S.data[SOURCE].iloc[source_ix]
    ks_rows,auc_rows=[] ,[]
    for target in TARGETS:
        for label in (0,1):
            a=source[source.label==label];b=S.data[target].query('label == @label')
            for f in NUMERIC:
                ks_rows.append({'source':SOURCE,'target':target,'class':label,'feature':f,
                    'KS_D':float(ks_2samp(a[f],b[f]).statistic),
                    'source_rows':len(a),'target_rows':len(b),
                    'scope':'14 numeric features, within label; no encoded TLD indicators'})
    for d,df in S.data.items():
        for f in ['URLLength','DomainLength','IsHTTPS','NoOfSubDomain']:
            auc=roc_auc_score(df.label,df[f])
            auc_rows.append({'dataset':d,'feature':f,'single_feature_AUROC':float(auc),
                             'orientation':'larger raw feature predicts phishing; no direction flipping'})
    S.table('10_class_conditional_numeric_KS',ks_rows)
    S.table('11_single_feature_AUROC',auc_rows)
    target=S.data['URL-Phish']
    D=[]
    for f in NUMERIC:
        D.append(float(ks_2samp(source[f],target[f]).statistic))
    src_rates=np.asarray(S.X['source_train'][:,14:].mean(axis=0)).ravel()
    tar_rates=np.asarray(S.X['URL-Phish'][:,14:].mean(axis=0)).ravel()
    D.extend(np.abs(src_rates-tar_rates).tolist())
    D=np.asarray(D)
    order=np.lexsort((np.arange(len(D)),-D))
    ranked=pd.DataFrame({'rank':range(1,len(D)+1),'position':order,
                         'feature':S.feature_labels[order],'KS_D':D[order]})
    S.table('13_encoded_marginal_KS_ranking',ranked)
    source_train_y=source.label.to_numpy()
    rows=[]
    for K in S.cfg['pruning_k']:
        effective=min(K,len(D)-1)
        keep=np.ones(len(D),bool);keep[order[:effective]]=False
        S.write_json(f'models/pruning_K{K}_mask.json',{'requested_K':K,'effective_K':effective,
                    'removed_columns':order[:effective],'retained_columns':np.flatnonzero(keep),
                    'selection':'unlabeled URL-Phish versus source-training marginal KS; post hoc diagnostic'})
        for model in ['LR','GB']:
            fit=S.models[model] if effective==0 else fit_checked(new_model(model,S.cfg),S.X['source_train'][:,keep],source_train_y,np.asarray(S.feature_safe)[keep])
            if effective:
                joblib.dump(fit,S.out/'models'/f'{model}_pruned_K{K}.joblib')
            for part in ['source_test']+TARGETS:
                d,ix=S.parts[part];df=S.data[d].iloc[ix]
                z=margin(fit,S.X[part] if effective==0 else S.X[part][:,keep]);p=expit(z)
                rows.append({'requested_K':K,'effective_K':effective,'retained_columns':int(keep.sum()),
                     'model':model,'dataset':d,**MetricCache(df.label.to_numpy(),p,ranking=z).values(),
                     'margin_q01':float(np.quantile(z,.01)),'margin_q50':float(np.median(z)),
                     'margin_q99':float(np.quantile(z,.99)),
                     'selection_scope':'target-feature-informed KS diagnostic; no target labels used to rank features'})
                S.save_predictions(f'pruning_K{K}_{model}_{part}',df.row_id,df.label,margin=z,probability=p)
        print(f'KS pruning K={K} (effective {effective})',flush=True)
    results=S.table('14_15_KS_pruning_numerical_results',rows)
    notes=[]
    for (model,d),block in results.groupby(['model','dataset']):
        block=block.sort_values('requested_K')
        for i in range(1,len(block)):
            a,b=block.iloc[i-1],block.iloc[i]
            notes.append({'model':model,'dataset':d,'from_K':int(a.requested_K),'to_K':int(b.requested_K),
                'delta_F1':b.F1-a.F1,'delta_positive_fraction':b.positive_fraction-a.positive_fraction,
                'delta_FPR':b.FPR-a.FPR,'delta_TPR':b.TPR-a.TPR,
                'interpretation':'measured change after refitting and removing the saved columns; no causal explanation inferred from a rebound'})
    S.table('S3_pruning_rebound_diagnostics',notes)
    S.done('KS_single_feature_AUC_and_pruning')

In [ ]:
execute_stage(S, stage_shift_and_pruning)

In [ ]:
def stage_lofo(S):
    seed_rows,summary=[] ,[]
    train_y=S.data[SOURCE].iloc[S.parts['source_train'][1]].label.to_numpy()
    predictions={}
    for seed in S.cfg['lofo_seeds']:
        baseline=S.models['GB'] if seed==S.cfg['seed'] else fit_checked(new_model('GB',S.cfg,seed),S.X['source_train'],train_y,S.feature_safe)
        if seed!=S.cfg['seed']:
            joblib.dump(baseline,S.out/'models'/f'GB_baseline_seed{seed}.joblib')
        for family,mask in S.masks.items():
            keep=~mask
            fit=fit_checked(new_model('GB',S.cfg,seed),S.X['source_train'][:,keep],train_y,np.asarray(S.feature_safe)[keep])
            joblib.dump(fit,S.out/'models'/f'GB_LOFO_{family}_seed{seed}.joblib')
            for part in ['source_test']+TARGETS:
                d,ix=S.parts[part];df=S.data[d].iloc[ix]
                zb=margin(baseline,S.X[part]);za=margin(fit,S.X[part][:,keep])
                pb,pa=expit(zb),expit(za)
                mb=MetricCache(df.label.to_numpy(),pb,ranking=zb)
                ma=MetricCache(df.label.to_numpy(),pa,ranking=za)
                predictions.setdefault((d,family),[]).append((mb,ma))
                base_f1,abl_f1=mb.values()['F1'],ma.values()['F1']
                seed_rows.append({'dataset':d,'family':family,'model':'GB','threshold':0.5,'seed':seed,'removed_columns':int(mask.sum()),
                    'retained_columns':int(keep.sum()),'F1_baseline':base_f1,'F1_ablated':abl_f1,
                    'delta_F1_ablated_minus_baseline':abl_f1-base_f1})
                S.save_predictions(f'LOFO_{family}_seed{seed}_{part}',df.row_id,df.label,
                                   baseline_margin=zb,ablated_margin=za,
                                   baseline_probability=pb,ablated_probability=pa)
        print(f'LOFO seed {seed}',flush=True)
    seeds=S.table('17_LOFO_all_seeds',seed_rows)
    for (d,family),pairs in predictions.items():
        part='source_test' if d==SOURCE else d
        _,ix=S.parts[part];df=S.data[d].iloc[ix]
        sampler=StratifiedRowResampler(df.label,df.url_group)
        rng=np.random.default_rng(S.cfg['seed']+6000)
        boot=[]
        for _ in range(S.cfg['bootstrap']):
            w=sampler.weights(rng)
            chosen=rng.integers(0,len(pairs),len(pairs))
            seed_deltas={j:pairs[j][1].f1_only(w)-pairs[j][0].f1_only(w) for j in np.unique(chosen)}
            boot.append(float(np.mean([seed_deltas[j] for j in chosen])))
        block=seeds[(seeds.dataset==d)&(seeds.family==family)]
        low,high=ci(boot)
        summary.append({'dataset':d,'family':family,'seeds':len(pairs),
             'removed_columns':int(S.masks[family].sum()),'retained_columns':int((~S.masks[family]).sum()),
             'F1_baseline_mean':block.F1_baseline.mean(),'F1_ablated_mean':block.F1_ablated.mean(),
             'delta_F1_mean':block.delta_F1_ablated_minus_baseline.mean(),
             'delta_F1_SD_across_seeds':block.delta_F1_ablated_minus_baseline.std(ddof=1),
             'delta_F1_ci_low':low,'delta_F1_ci_high':high,'CI_resamples':S.cfg['bootstrap'],
             'CI_estimand':'LightGBM raw p>=0.5; mean paired delta across seeds; seed/exact-URL cluster bootstrap',
             'delta_direction':'ablated minus baseline'})
        np.savez_compressed(S.out/'predictions'/f'LOFO_{d}_{family}_bootstrap.npz',mean_delta_draws=boot)
    S.table('17_LOFO_mean_delta_with_CI',summary)
    assert int((~S.masks['DomainProtocol']).sum())==9
    S.done('paired_LOFO_seed_and_row_uncertainty')

In [ ]:
execute_stage(S, stage_lofo)

In [ ]:
display(S.tables["17_LOFO_mean_delta_with_CI"])

## 8. MMD on a restricted URL-disjoint reference

M1/E4: the reference is source test after excluding URLs in either selected
target. Reference and targets each use one row per exact URL. This restriction
changes the reference population and must accompany any numerical claim.
Four gamma settings and matched-prior/class-conditional comparisons use the
14 numeric features, source-training preprocessing and explicit inference samples.


In [ ]:
def matched_mmd_size(requested,prior,availability):
    """FIX A04: Largest feasible integer-rounded matched-prior sample, >=2."""
    if not isinstance(requested,int) or requested<2 or not np.isfinite(prior) or not 0<=prior<=1:
        raise ValueError('MMD size/prior are invalid')
    a0,a1=map(int,availability)
    if min(a0,a1)<0:
        raise ValueError('MMD class availability cannot be negative')
    for n in range(min(requested,a0+a1),1,-1):
        n1=int(round(n*prior));n0=n-n1
        if n0<=a0 and n1<=a1:
            return n,n0,n1
    raise ValueError('No matched-prior MMD sample of at least two rows is available')

def permutation_pvalue(point,values):
    """FIX A05: Include numerical ties; different reduction orders round differently.

    RBF MMD is bounded in scale. A 64-epsilon tolerance protects ties, following
    SciPy's documented finite-precision permutation-test principle. Record the
    tolerance alongside the conservative +1 Monte Carlo p value.
    """
    values=finite_vector(values,'permutation statistics')
    if not np.isfinite(point):
        raise ValueError('Observed permutation statistic must be finite')
    tolerance=64*np.finfo(float).eps*max(1.,abs(point),float(np.max(np.abs(values))))
    return (1+int(np.sum(values>=point-tolerance)))/(1+len(values)),tolerance

def mmd_unbiased(Kxx,Kyy,Kxy):
    n,m=Kxx.shape[0],Kyy.shape[0]
    if min(n,m)<2:
        raise ValueError('Unbiased MMD requires at least two observations per domain')
    return float((Kxx.sum()-np.trace(Kxx))/(n*(n-1))+
                 (Kyy.sum()-np.trace(Kyy))/(m*(m-1))-2*Kxy.mean())

def mmd_permutation_values(K,labels,n,permutations,rng,stratify=False):
    """Balanced domain reassignment. Labels used only for the matched-prior control."""
    if K.shape!=(2*n,2*n):
        raise ValueError('Permutation implementation requires equal-sized samples')
    total,trace=float(K.sum()),float(np.trace(K))
    result=[]
    for first in range(0,permutations,32):
        batch=min(32,permutations-first)
        Z=np.empty((2*n,batch),dtype=float)
        for j in range(batch):
            z=-np.ones(2*n)
            if stratify:
                for label in np.unique(labels):
                    candidates=np.flatnonzero(labels==label)
                    count=int((labels[:n]==label).sum())
                    z[rng.choice(candidates,count,replace=False)]=1
            else:
                z[rng.choice(2*n,n,replace=False)]=1
            Z[:,j]=z
        q=np.sum(Z*(K@Z),axis=0)
        vals=((total+q)/2-trace)/(n*(n-1))-(total-q)/(2*n*n)
        result.extend(vals.tolist())
    return np.asarray(result)

def stratified_multinomial(labels,rng):
    w=np.zeros(len(labels))
    for label in np.unique(labels):
        ix=np.flatnonzero(labels==label)
        w[ix]=rng.multinomial(len(ix),np.full(len(ix),1/len(ix)))
    return w

def mmd_bootstrap_values(K,n,labels,replicates,rng):
    """Percentile bootstrap of the explicit inference sample, preserving class counts.

    U-statistic self terms exclude draws, not unique original indices. Repeated
    bootstrap copies of an original URL remain distinct draws in the statistic.
    Near the null this CI is descriptive; use the permutation test for H0.
    """
    Kxx,Kyy,Kxy=K[:n,:n],K[n:,n:],K[:n,n:]
    vals=[]
    for start in range(0,replicates,32):
        b=min(32,replicates-start)
        A=np.column_stack([stratified_multinomial(labels[:n],rng) for _ in range(b)])
        B=np.column_stack([stratified_multinomial(labels[n:],rng) for _ in range(b)])
        sx=np.sum(A*(Kxx@A),axis=0)-np.diag(Kxx)@A
        sy=np.sum(B*(Kyy@B),axis=0)-np.diag(Kyy)@B
        xy=np.sum(A*(Kxy@B),axis=0)
        vals.extend((sx/(n*(n-1))+sy/(n*(n-1))-2*xy/(n*n)).tolist())
    return np.asarray(vals)

def stage_mmd(S):
    train=S.data[SOURCE].iloc[S.parts['source_train'][1]].copy()
    # M1/E4/overlap review: use a common, exact-URL-disjoint restricted reference.
    # Removing target-shared URLs conditions this reference's distribution.
    reference=independent_reference(S)
    training_pool=train.drop_duplicates('url_group').reset_index(drop=True)
    pools={SOURCE:reference.drop_duplicates('url_group').reset_index(drop=True),
           **{d:S.data[d].drop_duplicates('url_group').reset_index(drop=True) for d in TARGETS}}
    S.table('M1_MMD_sampling_pool_priors',[
        {'dataset':d,'raw_pool_rows':len(reference if d==SOURCE else S.data[d]),
         'unique_URLs':len(pool),'raw_pool_phishing_fraction':float((reference if d==SOURCE else S.data[d]).label.mean()),
         'row_phishing_fraction':float(pool.label.mean()),
         'source_pool':'source test excluding URLs in either selected target' if d==SOURCE else 'development-disjoint target; one row per URL'} for d,pool in pools.items()])
    scaler=S.encoder.named_transformers_['numeric']
    X={d:scaler.transform(df[NUMERIC]) for d,df in pools.items()}
    rng=np.random.default_rng(S.cfg['seed']+7000)
    med_ix=rng.choice(len(training_pool),min(S.cfg['mmd_median_n'],len(training_pool)),replace=False)
    distances=pdist(scaler.transform(training_pool.iloc[med_ix][NUMERIC]),metric='sqeuclidean')
    positive=distances[distances>0]
    if not len(positive):
        raise ValueError('MMD median bandwidth undefined: all source numeric vectors coincide')
    gamma_med=1/(2*float(np.median(positive)))
    gammas={'1/d':1/len(NUMERIC),'median':gamma_med,
            'gamma_median/2':gamma_med/2,'2*gamma_median':gamma_med*2}
    source_prior=float(reference.label.mean())
    comparisons=[]
    for target in TARGETS:
        comparisons.extend([(target,'aggregate_empirical_prior',None,False),
                            (target,'aggregate_matched_source_prior',None,True),
                            (target,'legitimate_conditional',0,False),
                            (target,'phishing_conditional',1,False)])
    S.write_json('mmd/protocol.json',{'kernel':'exp(-gamma*||x-y||^2)',
        'estimator':'unbiased quadratic MMD squared; may be negative',
        'numeric_features':NUMERIC,'scaler_fit':'source training only',
        'sampling_unit':'one distinct exact URL; no replacement within a draw; domains not grouped',
        'restricted_reference_phishing_prior':source_prior,'gammas':gammas,
        'reference_partition':'source test excluding both selected target URL sets',
        'reference_scope':'restricted population; exclusion changes class mix and possibly features; not full source-test inference',
        'gamma_note':'half/double settings multiply gamma in exp(-gamma*d_squared), not the RBF length bandwidth',
        'bandwidth_fit_partition':'source training only',
        'median_rows':len(med_ix),'repeat_n_requested':S.cfg['mmd_n'],
        'inference_n_requested':S.cfg['mmd_inference_n'],
        'inference_note':'bootstrap CI and permutation p use a separate prespecified inference sample; repeat percentiles are stability intervals, not CIs of a mean'})
    S.write_json('mmd/median_sample_ids.json',training_pool.iloc[med_ix].row_id.tolist())
    def draw(target,klass,matched,n_requested,seed):
        rr=np.random.default_rng(seed)
        a,b=pools[SOURCE],pools[target]
        ia=np.arange(len(a)) if klass is None else np.flatnonzero(a.label.to_numpy()==klass)
        ib=np.arange(len(b)) if klass is None else np.flatnonzero(b.label.to_numpy()==klass)
        n=min(n_requested,len(ia),len(ib))
        if matched:
            avail=[min(int((a.label==k).sum()),int((b.label==k).sum())) for k in [0,1]]
            n,n0,n1=matched_mmd_size(n,source_prior,avail)
            def selected(df):
                return np.r_[rr.choice(np.flatnonzero(df.label.to_numpy()==0),n0,False),
                             rr.choice(np.flatnonzero(df.label.to_numpy()==1),n1,False)]
            sa,sb=selected(a),selected(b)
        else:
            if n<2:
                raise ValueError(f'{target} {klass}: MMD needs at least two rows per domain')
            sa,sb=rr.choice(ia,n,False),rr.choice(ib,n,False)
        return sa,sb
    repeats,inference,summary=[] ,[],[]
    for comp_id,(target,scope,klass,matched) in enumerate(comparisons):
        for repeat in range(S.cfg['mmd_repeats']):
            seed=S.cfg['seed']+8000+comp_id*100+repeat
            a,b=draw(target,klass,matched,S.cfg['mmd_n'],seed)
            xx,yy=X[SOURCE][a],X[target][b]
            Dxx,Dyy,Dxy=cdist(xx,xx,'sqeuclidean'),cdist(yy,yy,'sqeuclidean'),cdist(xx,yy,'sqeuclidean')
            stem=f'{target}_{scope}_repeat{repeat:02d}'
            np.savez_compressed(S.out/'mmd'/f'{stem}_ids.npz',
                source_ids=pools[SOURCE].iloc[a].row_id.to_numpy(str),
                target_ids=pools[target].iloc[b].row_id.to_numpy(str))
            for gamma_name,gamma in gammas.items():
                value=mmd_unbiased(np.exp(-gamma*Dxx),np.exp(-gamma*Dyy),np.exp(-gamma*Dxy))
                repeats.append({'target':target,'comparison':scope,'bandwidth':gamma_name,
                    'gamma':gamma,'repeat':repeat,'seed':seed,'n_source':len(a),'n_target':len(b),
                    'source_phishing_fraction':float(pools[SOURCE].iloc[a].label.mean()),
                    'target_phishing_fraction':float(pools[target].iloc[b].label.mean()),'MMD2':value})
        # Inference is explicit and distinct from the 20 stability draws.
        seed=S.cfg['seed']+9000+comp_id
        a,b=draw(target,klass,matched,S.cfg['mmd_inference_n'],seed)
        xx,yy=X[SOURCE][a],X[target][b];n=len(a)
        allx=np.vstack([xx,yy]);DD=cdist(allx,allx,'sqeuclidean')
        labels=np.r_[pools[SOURCE].iloc[a].label,pools[target].iloc[b].label]
        stem=f'{target}_{scope}_inference'
        np.savez_compressed(S.out/'mmd'/f'{stem}_ids.npz',
            source_ids=pools[SOURCE].iloc[a].row_id.to_numpy(str),
            target_ids=pools[target].iloc[b].row_id.to_numpy(str),labels=labels)
        for gamma_name,gamma in gammas.items():
            K=np.exp(-gamma*DD)
            point=mmd_unbiased(K[:n,:n],K[n:,n:],K[:n,n:])
            perms=mmd_permutation_values(K,labels,n,S.cfg['mmd_permutations'],
                np.random.default_rng(seed),stratify=matched)
            boot=mmd_bootstrap_values(K,n,labels,S.cfg['mmd_bootstrap'],np.random.default_rng(seed+100))
            low,high=ci(boot)
            pvalue,p_tolerance=permutation_pvalue(point,perms)
            safe=gamma_name.replace('/','_over_').replace('*','times')
            np.savez_compressed(S.out/'mmd'/f'{stem}_{safe}_inference_draws.npz',permutation_MMD2=perms,bootstrap_MMD2=boot)
            inference.append({'target':target,'comparison':scope,'bandwidth':gamma_name,'gamma':gamma,
                'inference_n_source':n,'inference_n_target':n,'MMD2':point,
                'MMD2_ci_low':low,'MMD2_ci_high':high,
                'permutation_p':pvalue,'p_tie_tolerance':p_tolerance,
                'permutations':len(perms),'bootstrap_resamples':len(boot),
                'permutation_scheme':'class-stratified domain reassignment' if matched else 'pooled domain reassignment',
                'CI_note':'conditional percentile bootstrap on this inference sample; near-null coverage is nonregular, use permutation test for H0'})
        print(f'MMD {target}: {scope}',flush=True)
    rep=S.table('07_08_09_MMD_all_repeats',repeats)
    for key,block in rep.groupby(['target','comparison','bandwidth'],sort=False):
        summary.append(dict(zip(['target','comparison','bandwidth'],key),
            gamma=block.gamma.iloc[0],n_source=block.n_source.iloc[0],n_target=block.n_target.iloc[0],
            repeats=len(block),MMD2_mean=block.MMD2.mean(),MMD2_SD=block.MMD2.std(ddof=1),
            MMD2_stability_p025=float(block.MMD2.quantile(.025)),
            MMD2_stability_p975=float(block.MMD2.quantile(.975)),
            interval_note='subsample spread, not CI of the mean'))
    S.table('07_08_09_MMD_stability_summary',summary)
    S.table('07_08_09_MMD_inference_CI_and_permutations',inference)
    order_rows=[]
    for (scope,bw,repeat),block in rep.groupby(['comparison','bandwidth','repeat']):
        values=block.set_index('target').MMD2
        order_rows.append({'comparison':scope,'bandwidth':bw,'repeat':repeat,
                           'LegitPhish_MMD2_exceeds_URL_Phish':bool(values['LegitPhish']>values['URL-Phish'])})
    S.table('M1_MMD_ordering_all_repeats',order_rows)
    ordering=pd.DataFrame(order_rows).groupby(['comparison','bandwidth'],as_index=False).agg(
        ordering_true_draws=('LegitPhish_MMD2_exceeds_URL_Phish','sum'),
        repeats=('repeat','count'),
        ordering_fraction=('LegitPhish_MMD2_exceeds_URL_Phish','mean'))
    S.table('M1_MMD_ordering_frequency',ordering)
    S.done('MMD_bandwidths_matched_priors_repeats_CIs_permutations')

In [ ]:
execute_stage(S, stage_mmd)

In [ ]:
display(S.tables["07_08_09_MMD_stability_summary"],
        S.tables["07_08_09_MMD_inference_CI_and_permutations"])

## 9. Error-focused explanations

**M4/R2-9:** use the baseline LightGBM to compare URL-Phish legitimate
false positives with correctly classified legitimate URLs. Save sampled row
IDs and raw-margin TreeSHAP contributions, verify additivity, and show both
source global importance and target error contrasts. Overlapping family sums
cannot be added together and do not establish causality.

In [ ]:
def tree_shap(S,X,ids,stem):
    contrib=S.models['GB'].booster_.predict(X,pred_contrib=True,num_threads=S.cfg['threads'])
    contrib=sparse.csr_matrix(contrib)
    if contrib.shape[1]!=X.shape[1]+1:
        raise AssertionError('Native LightGBM TreeSHAP must include its final expected-value column')
    reconstructed=np.asarray(contrib.sum(axis=1)).ravel()
    expected=margin(S.models['GB'],X)
    if not np.allclose(reconstructed,expected,rtol=1e-8,atol=1e-8):
        raise AssertionError('TreeSHAP additivity check failed on raw margins')
    sparse.save_npz(S.out/'shap'/f'{stem}_contributions_with_bias.npz',contrib)
    S.write_json(f'shap/{stem}_row_ids.json',list(ids))
    return contrib[:,:-1]

def stage_shap(S):
    rng=np.random.default_rng(S.cfg['seed']+10000)
    _,vaix=S.parts['source_validation'];va=S.data[SOURCE].iloc[vaix]
    selected=rng.choice(len(va),min(S.cfg['shap_source_n'],len(va)),False)
    values=tree_shap(S,S.X['source_validation'][selected],va.iloc[selected].row_id,
                     'source_validation_global')
    means=np.asarray(abs(values).mean(axis=0)).ravel()
    ranking=np.argsort(-means,kind='stable')
    S.table('03_source_global_SHAP',pd.DataFrame({'rank':range(1,len(means)+1),
        'position':ranking,'feature':S.feature_labels[ranking],'mean_absolute_SHAP_margin':means[ranking],
        'sample_rows':len(selected),'sample_partition':'source validation; identical sample for Figure 2'}))
    S.global_shap={'values':values,'sample_X':S.X['source_validation'][selected],
                   'selected_rows':selected,'ranking':ranking}
    errors,diagnostics=[] ,[]
    for d in TARGETS:
        df=S.data[d];y=df.label.to_numpy();_,p=S.scores['GB'][d]
        pools={'FP':np.flatnonzero((y==0)&(p>=.5)),'TN':np.flatnonzero((y==0)&(p<.5))}
        selected_groups,group_vals={} ,{}
        for group,pool in pools.items():
            chosen=rng.choice(pool,min(len(pool),S.cfg['shap_error_n']),False)
            selected_groups[group]=chosen
            if not len(chosen):
                diagnostics.append({'dataset':d,'group':group,'pool_rows':0,'SHAP_rows':0,
                    'status':'no such errors/controls; group contrast is undefined'})
                continue
            v=tree_shap(S,S.X[d][chosen],df.iloc[chosen].row_id,f'{d}_{group}')
            group_vals[group]={fam:np.asarray(v[:,mask].sum(axis=1)).ravel()
                              for fam,mask in S.masks.items()}
            https=np.asarray(v[:,S.feature_raw=='IsHTTPS'].sum(axis=1)).ravel()
            mx=v.max(axis=1).toarray().ravel()
            frac=float(np.mean((https>0)&np.isclose(https,mx,rtol=0,atol=1e-12)))
            diagnostics.append({'dataset':d,'group':group,'pool_rows':len(pool),'SHAP_rows':len(chosen),
                'status':'computed','HTTPS_top_positive_fraction':frac,
                'top_positive_definition':'IsHTTPS contribution > 0 and equal to largest feature contribution; positive ties included',
                'decision_rule':'raw GB probability >= 0.5',
                'SHAP_scale':'signed raw-margin TreeSHAP; families overlap and must not be summed together'})
        if set(group_vals)!={'FP','TN'}:
            continue
        samplers={g:StratifiedRowResampler(y[ix],df.iloc[ix].url_group) for g,ix in selected_groups.items()}
        for fam in S.masks:
            fp,tn=group_vals['FP'][fam],group_vals['TN'][fam]
            boot_fp,boot_tn,boot_delta=[] ,[],[]
            rr=np.random.default_rng(S.cfg['seed']+11000)
            for _ in range(S.cfg['bootstrap']):
                wf,wt=samplers['FP'].weights(rr),samplers['TN'].weights(rr)
                mf,mt=float(np.average(fp,weights=wf)),float(np.average(tn,weights=wt))
                boot_fp.append(mf);boot_tn.append(mt);boot_delta.append(mf-mt)
            lo,hi=ci(boot_delta);flo,fhi=ci(boot_fp);tlo,thi=ci(boot_tn)
            errors.append({'dataset':d,'family':fam,'FP_sample_rows':len(fp),'TN_sample_rows':len(tn),
                'FP_mean_signed_SHAP':float(fp.mean()),'FP_ci_low':flo,'FP_ci_high':fhi,
                'TN_mean_signed_SHAP':float(tn.mean()),'TN_ci_low':tlo,'TN_ci_high':thi,
                'FP_minus_TN_mean_signed_SHAP':float(fp.mean()-tn.mean()),
                'FP_minus_TN_ci_low':lo,'FP_minus_TN_ci_high':hi,'bootstrap_resamples':S.cfg['bootstrap'],
                'CI_estimand':'difference in signed family attribution among the saved sampled FP/TN rows',
                'interpretation':'error-subset association; not a causal explanation or test on unseen error subsets'})
    S.table('16_error_SHAP_family_contrasts_with_CI',pd.DataFrame(errors,columns=[
        'dataset','family','FP_sample_rows','TN_sample_rows','FP_mean_signed_SHAP',
        'FP_ci_low','FP_ci_high','TN_mean_signed_SHAP','TN_ci_low','TN_ci_high',
        'FP_minus_TN_mean_signed_SHAP','FP_minus_TN_ci_low','FP_minus_TN_ci_high',
        'bootstrap_resamples','CI_estimand','interpretation']))
    S.table('16_error_SHAP_sampling_and_HTTPS',diagnostics)
    S.done('matched_source_SHAP_figure_and_error_attributions')

In [ ]:
execute_stage(S, stage_shap)

## 10. Entropy, monitoring and selective prediction

M2/overlap audit: source-validation Platt logits supply entropy. MMD and entropy
share the restricted source reference; monitoring splits it into URL-disjoint
null/evaluation pools and uses distinct-URL empirical windows. This is not a
temporal deployment test. Selective curves retain row-level coverage and a
fixed source policy. Exact URL grouping does not establish domain independence.


In [ ]:
def weighted_entropy_stats(values,weights=None):
    values=np.asarray(values,float)
    w=np.ones(len(values)) if weights is None else np.asarray(weights,float)
    mean=float(np.average(values,weights=w))
    variance=float(np.average((values-mean)**2,weights=w))
    order=np.argsort(values,kind='stable');v=values[order];ow=w[order]
    positive=ow>0;v,ow=v[positive],ow[positive]
    cumulative=np.cumsum(ow);half=ow.sum()/2
    j=int(np.searchsorted(cumulative,half,side='left'))
    med=float((v[j]+v[j+1])/2) if cumulative[j]==half and j+1<len(v) else float(v[j])
    return {'mean':mean,'median':med,'variance':variance}

def bootstrap_entropy(values,labels,groups,replicates,seed):
    sampler=StratifiedRowResampler(labels,groups);rng=np.random.default_rng(seed)
    result={k:[] for k in ['mean','median','variance']}
    # Sorting once avoids re-sorting 100,000 rows for every median draw.
    order=np.argsort(values,kind='stable')
    ordered=np.asarray(values)[order]
    for _ in range(replicates):
        w=sampler.weights(rng)[order]
        mean=float(np.average(ordered,weights=w))
        var=float(np.average((ordered-mean)**2,weights=w))
        nonzero=w>0;vv,ww=ordered[nonzero],w[nonzero]
        cumulative=np.cumsum(ww);half=ww.sum()/2
        j=int(np.searchsorted(cumulative,half,side='left'))
        median=float((vv[j]+vv[j+1])/2) if cumulative[j]==half and j+1<len(vv) else float(vv[j])
        for k,v in [('mean',mean),('median',median),('variance',var)]:
            result[k].append(v)
    return result

def window_means(values,size,windows,rng):
    means=[]
    for first in range(0,windows,100):
        n=min(100,windows-first)
        ix=rng.integers(0,len(values),size=(n,size))
        means.extend(np.asarray(values)[ix].mean(axis=1).tolist())
    return np.asarray(means)

def stage_entropy_and_risk(S):
    # M2/overlap review: same restricted, distinct-URL reference as MMD.
    source_df=independent_reference(S)
    S.table('M2_reference_scope',[{'source_test_rows':len(S.parts['source_test'][1]),
        'restricted_distinct_URLs':len(source_df), 'restricted_phishing_fraction':float(source_df.label.mean()),
        'scope':'source-test URLs absent from both targets; restricted reference, not full source test'}])
    null_ix,eval_ix=target_halves(source_df,S.cfg['seed']+12000)
    save_split(S,'entropy_source_test_null_and_evaluation',source_df,
               {'null_threshold_pool':null_ix,'source_false_alarm_evaluation':eval_ix})
    S.entropy_data={}
    moments,alarms,deltas,risk=[] ,[],[],[]
    for model in ['LR','GB']:
        cal=S.source_calibrators[model,'Platt']
        source_z=margin(S.models[model],sparse.csr_matrix(S.encoder.transform(source_df[FEATURES])))
        source_h=stable_entropy(calibrated_logit(cal,source_z))
        null_h=source_h[null_ix]
        reference=source_df.iloc[eval_ix]
        datasets={SOURCE:(reference,source_h[eval_ix]),
                  **{d:(S.data[d].drop_duplicates('url_group'),
                       stable_entropy(calibrated_logit(cal,S.scores[model][d][0][
                           ~S.data[d].url_group.duplicated().to_numpy()]))) for d in TARGETS}}
        null_windows=window_means(null_h,S.cfg['window_size'],S.cfg['monitor_windows'],
                                 np.random.default_rng(S.cfg['seed']+13000))
        threshold=float(np.quantile(null_windows,.99))
        stat_boot={}
        for d,(df,H) in datasets.items():
            S.entropy_data[model,d]=(df,H)
            np.savez_compressed(S.out/'predictions'/f'entropy_{model}_{d}.npz',
                row_ids=df.row_id.to_numpy(str),entropy_bits=H)
            stats=weighted_entropy_stats(H)
            boot=bootstrap_entropy(H,df.label.to_numpy(),df.url_group.to_numpy(),
                                    S.cfg['bootstrap'],S.cfg['seed']+14000+(0 if d==SOURCE else TARGETS.index(d)+1))
            stat_boot[d]=boot
            row={'model':model,'dataset':d,'rows':len(H),'calibrator_fit':'source validation only; baseline model',
                 'entropy_scale':'bits, evaluated from calibrated logits',
                 'zero_entropy_fraction':float(np.mean(H==0)),
                 'entropy_q25':float(np.quantile(H,.25)),'entropy_q75':float(np.quantile(H,.75)),
                 'bootstrap_resamples':S.cfg['bootstrap']}
            for k,v in stats.items():
                row['entropy_'+k]=v
                row['entropy_'+k+'_ci_low'],row['entropy_'+k+'_ci_high']=ci(boot[k])
            moments.append(row)
            means=window_means(H,S.cfg['window_size'],S.cfg['monitor_windows'],
                              np.random.default_rng(S.cfg['seed']+15000+(0 if d==SOURCE else TARGETS.index(d)+1)))
            alarm=means>threshold;k=int(alarm.sum());n=len(alarm)
            low,high=binomial_ci(k,n)
            alarms.append({'model':model,'dataset':d,'window_size':S.cfg['window_size'],
                'null_pool_rows':len(null_h),'evaluation_pool_rows':len(H),
                'source_null_99pct_threshold':threshold,'windows':n,'alarms':k,
                'alarm_fraction':k/n,'alarm_percent':100*k/n,
                'alarm_fraction_ci_low':low,'alarm_fraction_ci_high':high,
                'alarm_percent_ci_low':100*low,'alarm_percent_ci_high':100*high,
                'CI_method':'exact binomial, conditional on fitted source threshold and empirical IID window generator',
                'target_labels_used_for_monitor':False,
                'operational_scope':'distinct-URL empirical IID windows; source reference excludes both target URL sets; temporal deployment/domain dependence not estimated'})
            np.savez_compressed(S.out/'predictions'/f'entropy_windows_{model}_{d}.npz',
                null_window_means=null_windows,evaluation_window_means=means,alarm=alarm,
                threshold=threshold,source_null_row_ids=source_df.iloc[null_ix].row_id.to_numpy(str))
        source_stats=weighted_entropy_stats(datasets[SOURCE][1])
        for d in TARGETS:
            target_stats=weighted_entropy_stats(datasets[d][1])
            row={'model':model,'target':d,'difference_direction':'target minus held-out source',
                 'bootstrap_resamples':S.cfg['bootstrap'],'CI_method':'separate class-stratified exact-URL bootstrap; restricted source reference'}
            for k in ['mean','median','variance']:
                point=target_stats[k]-source_stats[k]
                draw=np.asarray(stat_boot[d][k])-np.asarray(stat_boot[SOURCE][k])
                lo,hi=ci(draw)
                row['delta_entropy_'+k]=point
                row['delta_entropy_'+k+'_ci_low']=lo;row['delta_entropy_'+k+'_ci_high']=hi
            deltas.append(row)
        # Selective predictions use one source-validation policy at every coverage.
        t=S.source_thresholds[model,'Platt']
        for part in ['source_test']+TARGETS:
            d,ix=S.parts[part];df=S.data[d].iloc[ix];y=df.label.to_numpy()
            z=S.scores[model][part][0];p=calibrate(cal,z)
            H=stable_entropy(calibrated_logit(cal,z))
            # E2/M2: break exact entropy ties with a seeded key independent
            # of the label. Equal-URL copies share the hash; stable CSV row order
            # resolves residual ties. Coverage here is still measured in rows.
            tie=np.fromiter((int.from_bytes(hashlib.sha256(
                f'{S.cfg["seed"]+16000}:{group}'.encode()).digest()[:8], 'big')
                for group in df.url_group),dtype=np.uint64,count=len(H))
            order=np.lexsort((tie,H))
            S.write_json(f'predictions/selective_acceptance_order_{model}_{part}.json',
                         {'row_ids_low_to_high_entropy':df.iloc[order].row_id.tolist(),
                          'threshold':t,'threshold_provenance':'source validation F1 optimum',
                          'tie_breaking':'seeded exact-URL hash; stable CSV row order resolves identical hash/entropy ties'})
            S.save_predictions(f'selective_{model}_{part}',df.row_id,y,
                calibrated_probability=p,entropy_bits=H,decision=(p>=t).astype(np.int8))
            for numerator in range(20,1,-1):
                # FIX A09: Rational 5% grid prevents ceil(0.95*600) becoming
                # 571 from binary round-off; exact intended count is 570.
                coverage=numerator/20
                n=min(len(y),max(2,(numerator*len(y)+19)//20));accept=order[:n]
                accepted_y=y[accept];accepted_pred=p[accept]>=t
                # F1 stays defined if the most confident accepted subset has one class.
                f=float(f1_score(accepted_y,accepted_pred,zero_division=0))
                risk.append({'model':model,'dataset':d,'requested_coverage':coverage,
                    'achieved_coverage':n/len(y),'accepted_rows':n,'F1':f,
                    'accepted_phishing':int(accepted_y.sum()),
                    'accepted_legitimate':int((accepted_y==0).sum()),
                    'F1_zero_division_applied':bool(not accepted_y.any() and not accepted_pred.any()),
                    'classification_error':float(np.mean(accepted_pred!=accepted_y)),
                    'threshold':t,'threshold_provenance':'source validation only, fixed across coverage',
                    'selection':'calibrated baseline entropy; target labels used for evaluation only'})
    S.table('20_entropy_moments_with_CI',moments)
    S.table('20_entropy_target_minus_source_with_CI',deltas)
    S.table('20_entropy_window_alarms_with_CI',alarms)
    S.table('selective_risk_coverage',risk)
    S.done('source_calibrated_entropy_and_fixed_policy_selective_prediction')

In [ ]:
execute_stage(S, stage_entropy_and_risk)

In [ ]:
display(S.tables["20_entropy_moments_with_CI"],
        S.tables["20_entropy_window_alarms_with_CI"])

## 11. Figures and final validation

Figures, tables, predictions, models, prepared selections, URL group IDs and
input/environment records are saved in the new run directory. Completion means
numerical artifacts are available for review. It does not automatically approve
manuscript claims, upstream extraction definitions or submission readiness.


In [ ]:
def save_figure(S,name,fig):
    fig.savefig(S.out/'figures'/f'{name}.png',dpi=220,bbox_inches='tight')
    fig.savefig(S.out/'figures'/f'{name}.pdf',bbox_inches='tight')
    plt.close(fig)

def stage_figures(S):
    df=S.tables['14_15_KS_pruning_numerical_results']
    fig,axes=plt.subplots(1,2,figsize=(12,4.3),sharey=True)
    for ax,model in zip(axes,['LR','GB']):
        for d in [SOURCE]+TARGETS:
            a=df[(df.model==model)&(df.dataset==d)].sort_values('requested_K')
            ax.plot(a.requested_K,a.F1,marker='o',label=d)
        ax.set(title='Logistic regression' if model=='LR' else 'LightGBM',
               xlabel='Features removed by marginal KS',ylim=(0,1.03))
        ax.grid(alpha=.2);ax.legend(frameon=False)
    axes[0].set_ylabel('F1 at raw probability 0.5')
    fig.tight_layout();save_figure(S,'fig1_ks_pruning',fig)

    info=S.global_shap;rank=info['ranking'][:15][::-1]
    values=info['values'][:,rank].toarray();X=info['sample_X'][:,rank].toarray()
    fig,(left,right)=plt.subplots(1,2,figsize=(14,6.5),gridspec_kw={'width_ratios':[1,1.5]})
    left.barh(np.arange(len(rank)),np.mean(np.abs(values),axis=0),color='#3d6f9b')
    left.set(yticks=np.arange(len(rank)),yticklabels=S.feature_labels[rank],
             xlabel='Mean absolute TreeSHAP (raw margin)',title='Same source-validation sample')
    rr=np.random.default_rng(S.cfg['seed']+17000)
    for j in range(len(rank)):
        f=X[:,j];lo,hi=np.quantile(f,[.01,.99])
        color=np.clip((f-lo)/(hi-lo),0,1) if hi>lo else np.full(len(f),.5)
        sc=right.scatter(values[:,j],j+rr.uniform(-.25,.25,len(f)),c=color,
                         cmap='coolwarm',vmin=0,vmax=1,s=5,alpha=.55,rasterized=True)
    right.set(yticks=np.arange(len(rank)),yticklabels=S.feature_labels[rank],
              xlabel='Signed TreeSHAP contribution (raw margin)',title=f'Source validation: {len(values):,} saved rows')
    right.axvline(0,color='grey',linewidth=.8)
    bar=fig.colorbar(sc,ax=right,pad=.015);bar.set_ticks([0,1],labels=['Low','High']);bar.set_label('Feature value')
    fig.tight_layout();save_figure(S,'fig2_shap_source',fig)

    df=S.tables['selective_risk_coverage']
    fig,axes=plt.subplots(1,2,figsize=(12,4.3),sharey=True)
    for ax,model in zip(axes,['LR','GB']):
        for d in [SOURCE]+TARGETS:
            a=df[(df.model==model)&(df.dataset==d)].sort_values('achieved_coverage')
            ax.plot(a.achieved_coverage,a.F1,label=d)
        ax.set(title=model,xlabel='Coverage',ylim=(0,1.03));ax.legend(frameon=False);ax.grid(alpha=.2)
    axes[0].set_ylabel('F1 among accepted rows')
    fig.tight_layout();save_figure(S,'fig3_risk_coverage',fig)

    fig,axes=plt.subplots(1,2,figsize=(12,4.3),sharey=True)
    for ax,model in zip(axes,['LR','GB']):
        for color,d in zip(['#1f77b4','#ff7f0e','#2ca02c'],[SOURCE]+TARGETS):
            H=S.entropy_data[model,d][1]
            ax.hist(H,bins=np.linspace(0,1,51),density=True,histtype='step',linewidth=1.7,label=d,color=color)
            ax.axvline(H.mean(),linestyle='--',linewidth=1,alpha=.6,color=color)
        ax.set(title=f'{model}, source-validation Platt',xlabel='Predictive entropy (bits)')
        ax.legend(frameon=False);ax.grid(alpha=.15)
    axes[0].set_ylabel('Density')
    fig.tight_layout();save_figure(S,'fig4_entropy',fig)

    df=S.tables['21_label_budgets_mean_SD']
    fig,axes=plt.subplots(2,2,figsize=(12,8))
    for col,d in enumerate(TARGETS):
        for row,metric in enumerate(['Brier','ECE']):
            ax=axes[row,col]
            for model in ['LR','GB']:
                for method,style in [('Platt','-'),('isotonic','--')]:
                    a=df[(df.dataset==d)&(df.model==model)&(df.method==method)].sort_values('budget_fraction')
                    ax.errorbar(100*a.budget_fraction,a[metric+'_mean'],yerr=a[metric+'_SD'],
                                fmt='o'+style,capsize=3,label=f'{model} {method}')
            ax.set(title=d,xlabel='Labeled fraction of distinct target URLs (%)',ylabel=metric)
            ax.grid(alpha=.2);ax.legend(frameon=False,fontsize=9)
    fig.tight_layout();save_figure(S,'fig5_label_budget',fig)
    S.done('five_figures_from_saved_numerical_outputs')

FEATURE_DEFINITIONS=[
    (c, 'categorical' if c=='TLD' else 'numeric',
     'Imported unchanged from the unified CSV; exact upstream extraction rule '
     'requires separate attestation from Recomputed Feature and raw data')
    for c in FEATURES
]

PROTOCOL = '''# URL-grouped downstream analysis protocol

E1/E2 and overlap audit: inputs are the corrected full unified feature files.
Full rows are validated before applying fingerprinted prepared target selections.
Source partitions use the saved approximately 64/16/20 URL-grouped split; exact
sizes appear in 01_dataset_splits. All copies of an exact URL remain together.
Both main target cohorts exclude source training and validation URLs. The full
source test remains the classifier test; its target overlap is disclosed in
00b_target_cohort_audit. No feature extraction occurs in this notebook.

Exact URL identity strips surrounding whitespace only. URL hashes support group
checks without publishing raw URLs. Approved digests plus label/URL-length checks
support raw/unified alignment; exact upstream extraction definitions still need
separate attestation. Exact URL grouping does not establish domain independence.

E2/R1-2: target adaptation/evaluation folds are class-stratified by exact URL
group. Label budgets count distinct target URLs, with one representative row
per acquired URL. Fractions use the distinct-URL count of the selected target,
not all full-release rows. Labels guide simulated stratified acquisition; this
is not a blind real-world acquisition experiment. Evaluation retains its rows.
Repeated means/SD/percentiles describe split variability, not CIs of the mean.

E3/E4: AP consistently ranks raw model margins. AP_probability separately reports
the calibrated/raw probability ranking, retaining ties. Brier/ECE use probabilities;
F1 uses the explicitly recorded policy. Source policies, target-adapted policies
and evaluation-label oracle diagnostics remain distinct. Source-selected FPR
policies need not meet their budgets on evaluation data. Operating thresholds
are attainable score thresholds; raw margins avoid sigmoid saturation artifacts.

Intervals resample exact URL groups within class and retain all rows in a selected
group; metrics remain row-weighted. These are conditional on the fitted model and
fixed training data. Domains are not resampled. LOFO is LightGBM at raw p>=0.5;
its paired seed/URL-group bootstrap shares evaluation resamples across models and
seeds. Overlapping prespecified feature families do not form an additive partition.

M1/M2 reference correction: MMD and entropy use one common source-test reference
that excludes URLs appearing in either selected target. This restriction changes
the source class mix and potentially its feature distribution. Results therefore
describe that restricted reference, not the entire source-test population. The
reference and targets are deduplicated by exact URL for these analyses. Domain
dependence remains possible, so URL separation alone does not prove IID sampling.

MMD uses the 14 source-standardized numeric features only. Its scaler and median
gamma are learned on source training. Half/double settings multiply gamma in
exp(-gamma*distance_squared), not a conventional length bandwidth. The unbiased
MMD-squared estimator can be negative. Separate inference samples supply the
permutation tests and conditional bootstrap intervals; repeated subsample spreads
are stability intervals. Near-null bootstrap coverage is nonregular. Report the
actual permutation count/resolution and avoid universal bandwidth-ordering claims.

Entropy uses source-validation Platt logits. The restricted source reference is
split into URL-disjoint null/evaluation pools. Monitoring uses resampled empirical
IID distinct-URL windows and a source-null upper-mean threshold. Conditional
binomial intervals do not estimate temporal deployment or threshold uncertainty.
Selective prediction remains a row-coverage analysis of baseline models with a
fixed source-validation policy, independent of target labels for selection.

Target-label oracles, target-informed KS pruning and post-hoc family comparisons
are diagnostics. SHAP contrasts are associations, not proof of causation.
Actual LightGBM parameters are saved; subsample_freq=0 means row bagging is inactive.
Only a completed full run supplies numerical artifacts for author review. Completion
does not certify the manuscript, upstream definitions, or readiness for submission.
'''

def stage_finish(S):
    required=['00_label_and_extraction_sanity','04_feature_family_composition',
        '05_06_zero_shot_with_CI','18_operating_points_all_domains',
        '12_recalibration_all_repeats','12_recalibration_prespecified_split_CI',
        '21_label_budgets_all_draws','21_label_budgets_prespecified_draw_CI',
        '17_LOFO_mean_delta_with_CI','07_08_09_MMD_stability_summary',
        '07_08_09_MMD_inference_CI_and_permutations','16_error_SHAP_sampling_and_HTTPS',
        '20_entropy_moments_with_CI','20_entropy_target_minus_source_with_CI',
        '20_entropy_window_alarms_with_CI','14_15_KS_pruning_numerical_results']
    assert all(t in S.tables for t in required)
    if S.cfg['mode']=='full':
        assert '00b_target_cohort_audit' in S.tables
        assert 'target_cohort_audit' in S.manifest
        assert all((S.out/'splits'/name).is_file() for name in S.cfg['preparation_sha256'])
    op=S.tables['18_operating_points_all_domains']
    assert len(op)==2*3*len(S.cfg['fpr_budgets'])*2
    assert not op[['FPR_ci_low','FPR_ci_high','TPR_ci_low','TPR_ci_high']].isna().any().any()
    assert set(S.tables['12_recalibration_all_repeats'].dataset)==set(TARGETS)
    assert set(S.tables['21_label_budgets_all_draws'].dataset)==set(TARGETS)
    assert len(S.tables['07_08_09_MMD_inference_CI_and_permutations'])==8*4
    for i in range(1,6):
        assert list((S.out/'figures').glob(f'fig{i}_*.png'))
    definitions=[{'feature':a,'type':b,'definition':c,'missing_policy':
                 'No missing input cells accepted; exact extraction rules require upstream verification'}
                 for a,b,c in FEATURE_DEFINITIONS]
    # E1/R2-4: these CSV columns are known, but their upstream extraction
    # rules are not. Name this status table honestly until the separate
    # Recomputed Feature audit supplies exact definitions and raw URL checks.
    S.table('E1_15_feature_definition_status',definitions)
    matrix=[
        ('E1 / R1.1 / R2.4','unified input validation, source encoder and counts/masks','downstream implemented; upstream extraction unattested','00_label_and_extraction_sanity; 04_feature_family_composition; S2_*'),
        ('E2 / R1.2 / R2.6','disjoint calibration, ranking/threshold controls, repeated budgets on both targets','implemented','12_*; 21_*; E2_AP_*; splits/; models/; predictions/'),
        ('E3 / R1.3','attainable ROC policies, all domain/budget rows with CIs, saturation contributors','implemented','18_operating_points_all_domains; 19_probability_saturation; S1_LR_saturation_contributors'),
        ('E4 / R1.4 / R2.11','F1/AP, operating, LOFO, entropy and MMD uncertainty with explicit estimands','implemented','05_06_*; 12_*CI; 21_*CI; 17_*CI; 07_08_09_*CI*; 20_*CI*'),
        ('M1 / R2.5','MMD sizes/estimator/bandwidths, all repeats/orderings, permutation tests','implemented','07_08_09_*; M1_*; mmd/protocol.json'),
        ('M2','source-calibrated entropy, source threshold and source/target alarm intervals','implemented','20_*; predictions/entropy_*; fig4_entropy'),
        ('M3 / R2.1-3,7,10','two-benchmark scope, illustrative budgets, construction evidence limits','implementation controls added; manuscript wording remains author work','PROTOCOL.md; matched-prior MMD; all-domain results'),
        ('M4 / R2.9','error-subset SHAP and correctly defined single-feature AUROC','implemented when FP and TN exist; absence explicitly recorded','16_*; 11_single_feature_AUROC; shap/'),
        ('R2.8','LOFO deltas on both targets with paired uncertainty','implemented','17_LOFO_*; models/feature_masks.json'),
        ('M5 / R1.5','citation numbering, CNN reference, optional suggested literature','requires manuscript revision','No automatic citation or manuscript edits in this notebook'),
        ('M6 / R1.4','reproducibility artifacts and public release','artifacts implemented; publishing/archive identity remains author work','run_manifest.json; code/; splits/; models/; predictions/; RELEASE_CHECKLIST.md'),
        ('m1-2','percentage conversion and numerical direction of source-to-target changes','implemented as fresh numerical export; manuscript transfer required','manuscript_values.json; RESULT_NOTES.md'),
        ('m3','effect-size KS display without underflowed p values','implemented','10_class_conditional_numeric_KS; 13_encoded_marginal_KS_ranking'),
        ('m4','consistent terminology/spelling','feature names checked; extractor definitions and prose remain upstream/author work','E1_15_feature_definition_status; PROTOCOL.md'),
        ('m5','plain figure labels, pruning numerics and rebound evidence','implemented','figures/; 14_15_*; S3_pruning_rebound_diagnostics'),
        ('m6','validate all 100,000 original legitimate URL-Phish rows; disclose development-URL exclusion for main evaluation','full input checked; selected cohort separately audited','00_label_and_extraction_sanity; raw input SHA256 and row IDs'),
        ('m7','illustrative SOC budgets','explicitly documented; manuscript transfer required','PROTOCOL.md'),
        ('Editor closing request','marked manuscript and point-by-point response','requires author document revision','Notebook produces evidence; supplied documents were not edited')]
    S.table('REVIEWER_REQUIREMENTS',pd.DataFrame(matrix,columns=['reviewer_id','requirement','status','evidence']))
    protocol=PROTOCOL.replace('Twenty repeats',f'{S.cfg["repeats"]} repeats').replace(
        'five seeds',f'{len(S.cfg["lofo_seeds"])} seeds').replace('Twenty draws',f'{S.cfg["mmd_repeats"]} draws').replace(
        'n=3,000',f'n={S.cfg["mmd_n"]:,}').replace('n=1,000',f'n={S.cfg["mmd_inference_n"]:,}').replace(
        '1,000 class-stratified',f'{S.cfg["mmd_bootstrap"]:,} class-stratified').replace(
        '200 permutations',f'{S.cfg["mmd_permutations"]} permutations')
    (S.out/'PROTOCOL.md').write_text(protocol)
    (S.out/'FIGURE_CAPTIONS.md').write_text('''# Figure captions for this fresh run

1. Source-test and external-target F1 after removing columns ranked by marginal
   source-training versus unlabeled URL-Phish KS. All models refit on source
   training only. This is target-feature-informed post hoc pruning.
2. Native LightGBM TreeSHAP on the saved source-validation sample. Mean absolute
   contributions and the signed scatter use identical rows and raw-margin values.
   Feature colors scale between each feature's 1st/99th sample percentiles.
3. F1 against achieved coverage using baseline entropy from source-validation
   Platt calibration. Source-validation F1 thresholds stay fixed at all coverages;
   entropy ties are broken independently of labels. Curves are point estimates.
4. Entropy densities and mean lines from source-calibrated baseline models. The
   source pool is the restricted, distinct-URL source false-alarm evaluation half used in
   Table 20. Bins span 0 to 1 bit. Targets do not determine calibration or thresholds.
5. Held-out Brier/ECE against labeled fractions of distinct selected target URLs. Points are
   repeated-draw means; error bars are sample SD, not confidence intervals. Both
   external targets use disjoint adaptation/evaluation groups within each draw.
''')
    (S.out/'RELEASE_CHECKLIST.md').write_text('''# Author actions after the full run

- Use a completed full-mode run; smoke outputs are synthetic validation only.
- Transfer fresh values, CIs, row counts, feature counts and figures consistently.
- Preserve the interval estimands in PROTOCOL.md and state actual resample sizes.
- Explain the upstream native label mappings and URL feature definitions
  separately; this downstream notebook only verifies harmonized labels.
- Report the selected cohort sizes, exclusions and remaining source-test matches
  from 00b_target_cohort_audit. Full input counts and evaluation counts differ.
- Report the actual prepared source split sizes and exact-URL cluster uncertainty.
- State that MMD/entropy use the restricted source reference described in PROTOCOL;
  their source population differs from the full classifier source-test population.
- Keep target-label oracle diagnostics separate from source policies.
- Scope conclusions to the two benchmarks. Matched-prior/conditional MMD and SHAP
  are association evidence; avoid a causal construction claim.
- Audit Recomputed Feature independently; this notebook performs no extraction.
  If the paper reports Char-CNN results, reproduce them separately.
- Reconcile every reference number and the CNN citation; optional literature is
  an author/editor decision. Update the response and marked manuscript.
- Review redistribution licenses and URL sensitivity, then publish an archive
  with code, checksums, splits and permitted data. Add its actual link/DOI only
  after publication; do not leave fictitious GitHub/DOI placeholders.
''')
    values={'run_mode':S.cfg['mode'],'input_hashes':S.input_files,
            'tables':{k:df.to_dict('records') for k,df in S.tables.items()}}
    S.write_json('manuscript_values.json',values)
    notes=['# Data-derived notes for manuscript transfer','',
           'Both main target cohorts are development-URL-disjoint selections. Use their saved audit counts. MMD/entropy condition on a restricted source reference; exact URL grouping does not prove domain independence.','',
           'All values below come from this run. They do not establish a cause of historical anomalous operating points.','']
    zero=S.tables['05_06_zero_shot_with_CI']
    for model in ['LR','GB']:
        base=zero[(zero.model==model)&(zero.dataset==SOURCE)].iloc[0]
        for d in TARGETS:
            target=zero[(zero.model==model)&(zero.dataset==d)].iloc[0]
            delta=target.F1-base.F1
            notes.append(f'- {model}: source-test F1 {base.F1:.6f}; {d} F1 {target.F1:.6f}; target-minus-source {delta:+.6f}.')
    for _,r in S.tables['20_entropy_moments_with_CI'].iterrows():
        notes.append(f'- {r.model} {r.dataset}: entropy mean {r.entropy_mean:.6f}, median {r.entropy_median:.6f}, variance {r.entropy_variance:.6f}; exact-zero fraction {100*r.zero_entropy_fraction:.3f}%. Inspect its quartiles before explaining a median shift.')
    for _,r in S.tables['S3_pruning_rebound_diagnostics'].iterrows():
        if r.to_K==500:
            notes.append(f'- {r.model} {r.dataset}, K {r.from_K} to 500: F1 delta {r.delta_F1:+.6f}, positive-fraction delta {r.delta_positive_fraction:+.6f}, FPR delta {r.delta_FPR:+.6f}, TPR delta {r.delta_TPR:+.6f}. Removed columns and exact predictions are saved; this describes the rebound without assigning a causal mechanism.')
    (S.out/'RESULT_NOTES.md').write_text('\n'.join(notes)+'\n')
    # Hash every payload after all outputs have been written. The manifest and its
    # inventory are excluded to avoid circular hashes.
    inventory=[]
    for path in sorted(S.out.rglob('*')):
        if path.is_file() and path.name not in ('run_manifest.json','artifact_inventory.csv'):
            inventory.append({'path':str(path.relative_to(S.out)),
                              'bytes':path.stat().st_size,'sha256':digest_file(path)})
    pd.DataFrame(inventory).to_csv(S.out/'artifact_inventory.csv',index=False)
    S.manifest.update(status='complete',completed_stages=S.completed+['final_validation'],
                      finished_utc=datetime.now(timezone.utc).isoformat(),
                      output_count=len(inventory),notebook_outputs_safe_for_manuscript=False,
                      numerical_artifacts_ready_for_author_review=S.cfg['mode']=='full')
    S.write_json('run_manifest.json',S.manifest)
    print('COMPLETE:',S.out,flush=True)
    if S.cfg['mode']=='smoke':
        print('SMOKE MODE: validation only; do not use these numbers in the manuscript.',flush=True)
    return S.out

STAGES=[stage_inputs,stage_splits_and_models,stage_baseline_and_operating_points,
        stage_calibration,stage_label_budgets,stage_shift_and_pruning,stage_lofo,
        stage_mmd,stage_shap,stage_entropy_and_risk,stage_figures,stage_finish]

def run_all(config):
    verify_core_math()
    S=RunState(config)
    source_path=Path(__file__) if '__file__' in globals() else None
    if source_path is not None and source_path.is_file():
        (S.out/'code/reviewer_pipeline.py').write_bytes(source_path.read_bytes())
    for stage in STAGES:
        execute_stage(S,stage)
    return S

In [ ]:
execute_stage(S, stage_figures)

In [ ]:
execute_stage(S, stage_finish)

In [ ]:
from IPython.display import display,Markdown,Image
display(Markdown(f'**Status:** {S.manifest["status"]}  \n**Mode:** {S.cfg["mode"]}  \n**Run:** `{S.out}`'))
display(S.tables['REVIEWER_REQUIREMENTS'])
for _figure in sorted((S.out/'figures').glob('fig*.png')):
    display(Image(filename=str(_figure),width=900))

### Remaining interpretation limits

- Exact URL separation does not establish independence between domains.
- MMD/entropy describe a restricted source reference, not the full source test.
- Source-validation policies may exceed FPR budgets on held-out data.
- Fixed-training cluster intervals do not cover training-data uncertainty.
- Feature extraction and definitions remain a separate upstream audit.
- Reconcile manuscript, response, claims and public release with one fresh run.


In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import json

run_dir = Path(S.out)
manifest = json.loads((run_dir / "run_manifest.json").read_text())

if manifest["mode"] == "smoke":
    print("Synthetic smoke results validated; publication export skipped.")
else:

    assert manifest["status"] == "complete", "The full run has not completed."
    assert manifest["mode"] == "full", "Do not export synthetic smoke results."

    table_files = sorted((run_dir / "tables").glob("*.csv"))
    png_files = sorted((run_dir / "figures").glob("fig*.png"))
    pdf_files = sorted((run_dir / "figures").glob("fig*.pdf"))

    missing_tables = {
        f"{name}.csv" for name in S.tables
    } - {p.name for p in table_files}

    assert not missing_tables, f"Missing tables: {sorted(missing_tables)}"
    assert len(table_files) == len(S.tables), "Table count does not match this run."
    assert len(png_files) == 5 and len(pdf_files) == 5, "Expected five PNG/PDF figure pairs."
    assert {p.stem for p in png_files} == {p.stem for p in pdf_files}, (
        "PNG and PDF figure names do not match."
    )

    zip_path = run_dir.parent / f"{run_dir.name}_tables_and_figures.zip"
    files_to_export = table_files + png_files + pdf_files + [
        run_dir / "run_manifest.json",
        run_dir / "artifact_inventory.csv",
        run_dir / "FIGURE_CAPTIONS.md",
    ]

    with ZipFile(zip_path, "w", compression=ZIP_DEFLATED, compresslevel=6) as archive:
        for path in files_to_export:
            archive.write(path, arcname=str(path.relative_to(run_dir)))

    print(f"Verified {len(table_files)} tables and {len(png_files)} figures in PNG and PDF.")
    print("Completed run folder:", run_dir)
    print("Tables and figures ZIP:", zip_path)